# Spoiler-Erkennung in Filmrezensionen

**Lassen sich Spoiler allein an der Wortwahl erkennen? Ein Vergleich klassischer NLP-Verfahren**

Julius Drath · Roshan Raza Khalid · Konrad Krauth · Timon Gartung
Advanced Natural Language Processing · Prof. Dr. Kai Eckert · Technische Hochschule Mannheim

---

> **Forschungsfrage.** Kann man allein an der Wortwahl erkennen, ob eine Filmrezension die Handlung verrät, auch bei Filmen, die das Modell nie gesehen hat?

Die Frage hat zwei Teile, und beide werden getrennt beantwortet:

* **(a)** Reicht die Wortwahl? Operationalisiert als: Ein Bag-of-Words-Modell, das *nur* Wortvorkommen sieht, schlägt die Baselines statistisch signifikant.
* **(b)** Auch bei unbekannten Filmen? Operationalisiert als: Das gilt auch, wenn kein Testfilm im Training vorkommt (Aufteilung nach Film).

### Hypothesen (vorab festgelegt, aus der Präsentation)

| | Hypothese | Prüfung | Kapitel |
|---|---|---|---|
| **H1** | *Handlung statt Meinung.* Das Modell nutzt Handlungswörter („dies“, „reveals“) als Spoiler-Hinweise und Bewertungswörter („acting“, „recommend“) als Nicht-Spoiler-Hinweise. | Vorab definierte Wortlisten; Vergleich der Modellgewichte beider Listen mit einem Mann-Whitney-U-Test | 7.1 |
| **H2** | *Unbekannte Filme sind schwerer*, weil Figurennamen nicht mehr helfen. | (a) Gepaarter Vergleich zufällige vs. filmweise Aufteilung über mehrere Seeds; (b) Mechanismus: Namen entfernen und prüfen, ob der Abstand schrumpft | 7.2 |
| **H3** | *Typische Fehlalarme.* Wörter wie „killed“ oder „murder“ lösen Alarm aus, auch wenn nichts verraten wird. | Längenadjustiertes Fehlalarm-Verhältnis mit Bootstrap-Konfidenzintervall | 7.3 |

### Aufbau

| Kapitel | Inhalt |
|---|---|
| 1 | Fehleranalyse des bisherigen Ansatzes |
| 2 | Daten laden und bereinigen |
| 3 | Datenverständnis (EDA) |
| 4 | Methodik: Aufteilungen, Pipeline, Metriken, Validierung |
| 5 | Modellvergleich mit Kreuzvalidierung und Hyperparameter-Tuning |
| 6 | Evaluation des besten Modells: Kurven, Konfidenzintervalle, Signifikanz, Merkmalsgewichte, Störfaktor Länge |
| 7 | Prüfung der Hypothesen H1 bis H3 |
| 8 | Zusatzanalysen: Figurennamen, Genres, Einzelfilme, Handlungsabgleich, Label-Qualität, Nutzer-Split |
| 9 | Antwort auf die Forschungsfrage und Kernergebnisse |
| 10 | Limitationen |
| 11 | Gutachter-Checkliste |
| 12 | Exportierte Abbildungen und Tabellen |

### Hinweise zur Ausführung

* **Daten:** `IMDB_movie_details.json` in den Projektordner `data/` legen. Fehlt `IMDB_reviews.json`, wird sie ?ber KaggleHub dorthin heruntergeladen und f?r sp?tere L?ufe gespeichert (Kaggle-Zugriff erforderlich).
* **Pakete:** `numpy`, `pandas`, `scipy`, `scikit-learn` (≥ 1.2), `matplotlib`, `joblib`, `kagglehub[pandas-datasets]`. Optional: `shap` (Kapitel 6.5), `tabulate` (Markdown-Tabellen).
* **Laufzeit:** Vollständiger Lauf mit 100.000 Rezensionen und 3-facher Kreuzvalidierung: **etwa 60 bis 90 Minuten** auf einem Laptop. Zellen mit langer Laufzeit sind mit ⏳ markiert. Für einen Probelauf `FAST_MODE = True` setzen (etwa 15 Minuten, kleinere Stichprobe, weniger Bootstrap-Wiederholungen).
* **Reproduzierbarkeit:** Alle Zufallsentscheidungen hängen an `RANDOM_STATE`. Zwischenergebnisse der Textvektorisierung werden in `cache/` abgelegt.
* **Export:** Abbildungen landen als PNG mit 300 dpi in `figures/`, Tabellen als CSV (und Markdown) in `tables/`.

## 1 · Fehleranalyse des bisherigen Ansatzes

Grundlage ist der vorherige Durchlauf auf den echten Daten (100.000 Rezensionen). Die Prüfung erfolgt strikt aus Gutachtersicht: Jeder Punkt nennt den Befund, warum er die Ergebnisse angreifbar macht, und wo dieses Notebook ihn behebt.

| # | Befund im bisherigen Ansatz | Warum das die Ergebnisse angreifbar macht | Behebung |
|---|---|---|---|
| 1 | **Zielvariable im Text.** Die stärksten Merkmale waren `spoilers` (Gewicht +14,6) und `spoiler` (+13,6), gefolgt von `alert`. Das Modell lernte zum Teil die *Selbstdeklaration* der Verfasser („This review contains spoilers“), nicht die Wortwahl über die Handlung. | Das Label stammt vom Verfasser; wer es setzt, schreibt oft auch die Warnung. Das ist eine Form von Label-Leakage über die Merkmale. Die Forschungsfrage fragt nach der *Wortwahl über die Handlung*, nicht nach Warnhinweisen. Der gemessene Effekt war klein (−0,009 F1), aber unkontrolliert im Hauptmodell. | Kap. 4: Warnmarker (`spoil…`, „spoiler alert“) werden **innerhalb der Pipeline** entfernt. Das ist das Hauptmodell. Kap. 8.5: Ablation mit Markern. |
| 2 | **Willkürliche Entscheidungsregel für H2.** Δ F1 = −0,007 wurde als „kein Unterschied“ gewertet, weil eine feste Schwelle von 0,01 gesetzt war. Dabei war der Effekt in allen 6 Modell-Merkmal-Kombinationen und in allen 3 Seeds negativ. | Eine Hypothese darf nicht an einer ad hoc gewählten Schwelle scheitern. Ein konsistenter kleiner Effekt ist ein Effekt. Ohne Test bleibt unklar, ob er zufällig ist. | Kap. 7.2: gepaarter t-Test und Wilcoxon-Test über 5 Seeds, Konfidenzintervall der Differenz. Verdikt nach Konfidenzintervall statt nach Schwelle. |
| 3 | **Keine Konfidenzintervalle, keine Signifikanztests.** Alle Vergleiche (Modelle, Genres, Einzelfilme, Handlungsabgleich) beruhten auf Punktschätzern aus einem Testlauf. | Unterschiede von 0,01 bis 0,03 F1 sind bei 500 bis 15.000 Testrezensionen oft innerhalb des Rauschens. Das Einzelfilm-Ergebnis „+0,033 F1, hilft deutlich“ war so nicht belegt. | Kap. 6.3: Bootstrap-Konfidenzintervalle (gepaart) für alle Hauptmetriken; Kap. 8.3: wiederholte Aufteilungen pro Film. |
| 4 | **Keine Kreuzvalidierung.** Hyperparameter wurden auf einer einzigen Validierungsmenge gewählt; Streuung der Modellgüte war unbekannt. | Ein Lauf kann Glück oder Pech sein. Ohne Varianzschätzung lässt sich „Modell A ist besser als B“ nicht belegen. | Kap. 5: `GridSearchCV` mit 3-facher Kreuzvalidierung (stratifiziert bzw. gruppiert nach Film), Mittelwert ± Standardabweichung. |
| 5 | **Vorverarbeitung außerhalb einer Pipeline.** Vektorisierung war manuell auf Training gefittet (korrekt), aber nicht als `Pipeline` gekapselt; bei Kreuzvalidierung wäre Leakage leicht passiert. | Jede Transformation, die vor dem Split auf alle Daten schaut (Vokabular, IDF, Namenslisten), verrät Testinformation. | Kap. 4: alle lernenden Schritte (TF-IDF, SVD) in `sklearn.Pipeline`, Fit ausschließlich auf Trainingsfalten. |
| 6 | **Keine Schwellenwahl.** Lineare Modelle mit `class_weight="balanced"` und Standardschwelle 0 → Precision 0,47, Recall 0,61. Die Schwelle wurde nie auf die Zielmetrik eingestellt. | F1 hängt stark von der Schwelle ab; ein Vergleich bei unkalibrierten Schwellen vergleicht Zufall. | Kap. 5: Schwelle auf der Validierungsmenge gewählt, nie auf der Testmenge. AP und ROC-AUC als schwellenfreie Metriken. |
| 7 | **Störfaktor Textlänge nicht kontrolliert.** Spoiler-Rezensionen sind im Median 260 Wörter lang, Nicht-Spoiler 170. | Ein Modell kann „lang = Spoiler“ lernen. Das wäre keine Erkennung an der Wortwahl. | Kap. 6.6: Baseline nur aus Länge; Modellgüte je Längenquartil; H3 längenadjustiert. |
| 8 | **Duplikate nur über (Film, Nutzer, Text) entfernt.** Identische Texte unter anderen Filmen oder Nutzern blieben erhalten. | Ein Text in Training und Test ist direkte Leakage. | Kap. 2: zusätzliche Deduplizierung über den Text selbst, Anzahl wird berichtet. |
| 9 | **Nutzerüberlappung zwischen Training und Test** nicht geprüft. Derselbe Autor kann in beiden Mengen sein. | Ein Modell kann den Stil eines Autors statt der Spoiler-Sprache lernen (Stil-Leakage). | Kap. 4.1: Überlappung gemessen; Kap. 8.6: Sensitivitätsanalyse mit Aufteilung nach Nutzer. |
| 10 | **Namenserkennung zu streng.** „Joker“, „Dent“, „Two Face“ wurden nicht als Namen erkannt (weil die Wörter irgendwo kleingeschrieben vorkamen); „0 Namen in Top 50“ war deshalb unterschätzt. | Der Mechanismus-Test für H2 hängt von der Namensliste ab. | Kap. 2.3: Heuristik nutzt das Verhältnis Groß- zu Kleinschreibung statt eines harten Ausschlusses. Limitation bleibt und wird benannt. |
| 11 | **H1 mit vier handverlesenen Wörtern geprüft** („4 von 4 passen“). | Vier Beispiele sind keine Hypothesenprüfung, sondern Anekdote. | Kap. 7.1: zwei vorab definierte Wortlisten (je 30 Wörter), Mann-Whitney-U-Test auf den Gewichten. |
| 12 | **Verdikte mit ad-hoc-Schwellen** auch bei Genres, Label-Qualität, Handlungsabgleich. | Nicht nachvollziehbar, nicht replizierbar. | Alle Verdikte folgen jetzt aus Konfidenzintervallen oder Tests; die Regel steht jeweils im Code. |
| 13 | **`LinearSVC` ohne `random_state`.** | Nicht exakt reproduzierbar. | Gesetzt. |
| 14 | **Kein Baum-Ensemble** im Vergleich. | Ein Modellvergleich nur innerhalb linearer Modelle ist unvollständig. | Kap. 5: Gradient Boosting auf SVD-reduzierten TF-IDF-Merkmalen. |

**Was korrekt war und beibehalten wird:** Vektorisierung nur auf Trainingsdaten gefittet; stratifizierte zufällige Aufteilung; gruppierte Aufteilung nach Film ohne Überlappung (0 gemeinsame Filme, geprüft); Testmenge erst für die Endbewertung benutzt; Modellauswahl auf der Validierungsmenge; Metriken für die Minderheitsklasse (Precision, Recall, F1, AP) statt Accuracy; Handlungstexte und Namenslisten enthalten keine Labels (kein Leakage).

## 0 · Setup und Konfiguration

In [1]:
import re
import time
import textwrap
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from joblib import Memory
from scipy import stats

from sklearn.base import clone
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, GroupKFold, GroupShuffleSplit, StratifiedKFold,
                                     train_test_split)
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11, "font.size": 10})
DARK, LIGHT, ACCENT, GREY, GREEN = "#001a4d", "#7aa6d6", "#e07a5f", "#b0b7c3", "#3a8f5c"

In [2]:
# ---------------- Konfiguration ----------------
FAST_MODE = True            # True = schneller Probelauf (~15 min), False = vollständiger Lauf (~60-90 min)
RANDOM_STATE = 42

def find_project_root(start: Path) -> Path:
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file():
            return folder
    raise FileNotFoundError("Projektstamm mit pyproject.toml nicht gefunden.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
TAB_DIR = PROJECT_ROOT / "outputs" / "tables"
CACHE_DIR = PROJECT_ROOT / "cache"
for d in (DATA_DIR, FIG_DIR, TAB_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 30_000 if FAST_MODE else 100_000   # Stichprobe für die Modellexperimente
VAL_SIZE, TEST_SIZE = 0.15, 0.15                  # Rest (70 %) ist Training
N_FOLDS = 2 if FAST_MODE else 3                   # Kreuzvalidierung innerhalb des Trainings
N_SEEDS = 3 if FAST_MODE else 5                   # Wiederholungen für H2
N_BOOT = 300 if FAST_MODE else 1000               # Bootstrap-Wiederholungen
N_JOBS = 1                                        # parallele CV-Fits (mehr = schneller, mehr RAM)
SVD_DIM = 200 if FAST_MODE else 300               # Dimension für das Baum-Ensemble
N_GENRES, N_FILMS = 5, 3
FILM_GENERAL_N = 15_000 if FAST_MODE else 30_000
MIN_GENRE_TEST = 200

VECTORIZER_ARGS = dict(token_pattern=r"(?u)\b[a-zA-Z]{2,}\b", ngram_range=(1, 2),
                       min_df=5, max_df=0.9, max_features=100_000)
MEMORY = Memory(CACHE_DIR, verbose=0)
rng_global = np.random.default_rng(RANDOM_STATE)
findings = {}   # Verdikte für Kapitel 9
exports = []    # Liste exportierter Dateien für Kapitel 12
print(f"Modus: {'SCHNELL' if FAST_MODE else 'VOLLSTÄNDIG'} · Stichprobe {SAMPLE_SIZE:,} · {N_FOLDS}-fache CV · {N_BOOT} Bootstraps")

Modus: SCHNELL · Stichprobe 30,000 · 2-fache CV · 300 Bootstraps


In [3]:
# ---------------- Hilfsfunktionen ----------------
def de(number, decimals=0):
    text = f"{number:,.{decimals}f}"
    return text.replace(",", "X").replace(".", ",").replace("X", ".")


def note(text):
    # Interpretationstext direkt unter Grafik oder Tabelle, aus den tatsächlichen Zahlen erzeugt
    display(Markdown("> **Interpretation.** " + text))


def save_fig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    exports.append(str(path))
    plt.show()


def save_table(df, name):
    path = TAB_DIR / f"{name}.csv"
    df.to_csv(path, float_format="%.4f")
    exports.append(str(path))
    try:
        (TAB_DIR / f"{name}.md").write_text(df.to_markdown(floatfmt=".3f"), encoding="utf-8")
    except ImportError:
        pass
    return df


def fmt_ci(point, lo, hi, decimals=3):
    return f"{point:.{decimals}f} [{lo:.{decimals}f}; {hi:.{decimals}f}]"


def score_of(estimator, X):
    # Kontinuierlicher Spoiler-Score: Abstand zur Entscheidungsgrenze bzw. Wahrscheinlichkeit
    if hasattr(estimator, "decision_function"):
        return np.asarray(estimator.decision_function(X))
    return np.asarray(estimator.predict_proba(X))[:, 1]


def best_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return float(thresholds[np.argmax(f1[:-1])])


def metrics(y_true, y_pred, y_score):
    return {"Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "AP": average_precision_score(y_true, y_score),
            "ROC-AUC": roc_auc_score(y_true, y_score)}


def bootstrap_metrics(y_true, models, n_boot=None, seed=RANDOM_STATE):
    # models: {name: (scores, preds)}. Gleiche Resamples für alle Modelle -> gepaarte Vergleiche möglich.
    n_boot = N_BOOT if n_boot is None else n_boot
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    out = {name: {"F1": [], "AP": [], "ROC-AUC": []} for name in models}
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yi = y_true[idx]
        if yi.min() == yi.max():
            continue
        for name, (scores, preds) in models.items():
            out[name]["F1"].append(f1_score(yi, np.asarray(preds)[idx], zero_division=0))
            out[name]["AP"].append(average_precision_score(yi, np.asarray(scores)[idx]))
            out[name]["ROC-AUC"].append(roc_auc_score(yi, np.asarray(scores)[idx]))
    return {name: {m: np.array(v) for m, v in d.items()} for name, d in out.items()}


def ci(values, level=0.95):
    lo, hi = np.quantile(values, [(1 - level) / 2, 1 - (1 - level) / 2])
    return float(lo), float(hi)

## 2 · Daten laden und bereinigen

### 2.1 Laden

Zwei Dateien im JSON-Lines-Format: eine Zeile pro Rezension (Text, Film-ID, Nutzer-ID, Datum, Bewertung, Label `is_spoiler`) und eine Zeile pro Film (Genres, Handlungszusammenfassung `plot_summary`, ausführliche Handlung `plot_synopsis`).

In [4]:
REVIEWS_FILE, MOVIES_FILE = DATA_DIR / "IMDB_reviews.json", DATA_DIR / "IMDB_movie_details.json"

start = time.time()
if not REVIEWS_FILE.exists():
    print("IMDB_reviews.json fehlt – lade die Datei nach data/ …")
    import kagglehub

    print("Arbeitsverzeichnis:", Path.cwd())
    print("Gesuchter Pfad:", REVIEWS_FILE.resolve())
    print("Datei vorhanden:", REVIEWS_FILE.exists())
    print("KaggleHub-Version:", kagglehub.__version__, flush=True)

    start = time.time()
    print("Download startet …", flush=True)

    result = kagglehub.dataset_download(
        "rmisra/imdb-spoiler-dataset",
        path="IMDB_reviews.json",
        output_dir=str(DATA_DIR),
    )

    print(f"Download beendet nach {time.time() - start:.1f} s: {result}", flush=True)
reviews = pd.read_json(
    REVIEWS_FILE,
    lines=True,
    convert_dates=False,
    dtype={"movie_id": str, "user_id": str},
)

if not MOVIES_FILE.exists():
    raise FileNotFoundError(f"Bitte IMDB_movie_details.json in '{DATA_DIR}/' ablegen.")

movies = pd.read_json(MOVIES_FILE, lines=True, convert_dates=False, dtype={"movie_id": str})
print(f"Geladen in {time.time() - start:.0f} s: {de(len(reviews))} Rezensionen, {de(len(movies))} Filme")
reviews.head(3)

C:\Users\timon\AppData\Local\Temp\ipykernel_14504\1750989688.py:23: Pandas4Warning: The 'convert_dates' keyword in read_json is deprecated and will be removed in a future version. Pass dtype=False to disable type conversion, or parse date columns with pd.to_datetime after reading.
  reviews = pd.read_json(


Geladen in 8 s: 573.913 Rezensionen, 1.572 Filme


C:\Users\timon\AppData\Local\Temp\ipykernel_14504\1750989688.py:33: Pandas4Warning: The 'convert_dates' keyword in read_json is deprecated and will be removed in a future version. Pass dtype=False to disable type conversion, or parse date columns with pd.to_datetime after reading.
  movies = pd.read_json(MOVIES_FILE, lines=True, convert_dates=False, dtype={"movie_id": str})


,review_date,movie_id,user_id,is_spoiler,review_text,rating,review_summary
0,10 February 2006,tt0111161,ur1898687,True,"In its Oscar year, Shawshank Redemption (written and directed by Frank Darabont, after the novella Rita Hayworth and...",10,A classic piece of unforgettable film-making.
1,6 September 2000,tt0111161,ur0842118,True,The Shawshank Redemption is without a doubt one of the most brilliant movies I have ever seen. Similar to The Green ...,10,Simply amazing. The best film of the 90's.
2,3 August 2001,tt0111161,ur1285640,True,"I believe that this film is the best story ever told on film, and I'm about to tell you why.Tim Robbins plays Andy D...",8,The best story ever told on film


### 2.2 Bereinigung

Entscheidungen und Begründung:

* **Leere Texte** entfernen: Ohne Text gibt es keine Wortwahl.
* **Exakte Textduplikate** entfernen, über alle Filme und Nutzer hinweg (Punkt 8 der Fehleranalyse): Derselbe Text in Training und Test wäre Leakage.
* **Rezensionen ohne Filmdatensatz** entfernen: Für Genres und Figurennamen werden die Filmdaten benötigt.
* **Keine Ausreißer entfernen:** Sehr lange Rezensionen sind echte Daten, kein Messfehler (Kap. 3.3).

In [ ]:
n_raw = len(reviews)
movies = movies.drop_duplicates("movie_id").reset_index(drop=True)
movies["genre"] = movies["genre"].apply(lambda g: list(g) if isinstance(g, list) else [])
for col in ("plot_summary", "plot_synopsis"):
    movies[col] = movies[col].fillna("").astype(str)
movies["plot_text"] = (movies["plot_summary"] + " " + movies["plot_synopsis"]).str.strip()

reviews["review_text"] = reviews["review_text"].fillna("").astype(str)
reviews["label"] = reviews["is_spoiler"].astype(int)
steps = [("Rohdaten", len(reviews))]
reviews = reviews[reviews["review_text"].str.strip().str.len() > 0]
steps.append(("ohne leere Texte", len(reviews)))
reviews = reviews.drop_duplicates(subset=["movie_id", "user_id", "review_text"])
steps.append(("ohne Duplikate (Film, Nutzer, Text)", len(reviews)))
reviews = reviews.drop_duplicates(subset=["review_text"])
steps.append(("ohne Textduplikate (über alle Filme/Nutzer)", len(reviews)))
reviews = reviews[reviews["movie_id"].isin(movies["movie_id"])]
steps.append(("nur Filme mit Filmdatensatz", len(reviews)))

reviews["review_date"] = pd.to_datetime(reviews["review_date"], format="%d %B %Y", errors="coerce")
reviews["year"] = reviews["review_date"].dt.year
reviews["rating"] = pd.to_numeric(reviews["rating"], errors="coerce")
reviews["n_words"] = reviews["review_text"].str.count(r"\S+")
reviews = reviews.merge(movies[["movie_id", "genre"]], on="movie_id", how="left").reset_index(drop=True)

cleaning = pd.DataFrame(steps, columns=["Schritt", "Rezensionen"]).set_index("Schritt")
cleaning["Entfernt"] = -cleaning["Rezensionen"].diff().fillna(0).astype(int)
save_table(cleaning, "02_bereinigung")

### 2.3 Figurennamen aus den Handlungstexten

Für H2 und Kapitel 8.1 wird pro Film eine Liste von Figurennamen benötigt. Der Datensatz liefert keine, deshalb werden sie aus den Handlungstexten abgeleitet. **Warum diese Heuristik?** Ein Named-Entity-Recognizer würde eine weitere Abhängigkeit einführen und auf Fantasienamen („Anduril“) schlecht funktionieren; die Handlungstexte enthalten die Namen ohnehin.

1. Kandidaten: großgeschriebene Wörter mit mindestens 3 Buchstaben, die **nicht am Satzanfang** stehen.
2. Ein Kandidat gilt als Name, wenn er in den Handlungstexten häufiger groß (mitten im Satz) als klein geschrieben wird. Das ist die Korrektur zu Punkt 10 der Fehleranalyse: „Joker“ bleibt ein Name, auch wenn „joker“ einmal vorkommt.
3. Namen, die in mehr als 3 % aller Filme vorkommen („American“, „Christmas“), sind nicht filmspezifisch und werden entfernt.

Die Namenslisten enthalten **keine Labels** und stammen nicht aus den Rezensionen; sie verursachen keine Leakage.

In [ ]:
CAP_MID = re.compile(r"(?<![.!?]\s)(?<!^)\b[A-Z][a-zA-Z]{2,}\b")
LOWER_WORD = re.compile(r"\b[a-z]{3,}\b")
MAX_NAME_MOVIE_SHARE = 0.03

lower_counts, cap_counts = Counter(), Counter()
for text in movies["plot_text"]:
    lower_counts.update(LOWER_WORD.findall(text))
    cap_counts.update(w.lower() for w in CAP_MID.findall(text))
name_like = {w for w, c in cap_counts.items() if c > lower_counts.get(w, 0)}

candidates = movies["plot_text"].apply(lambda t: {w for w in CAP_MID.findall(t) if w.lower() in name_like})
movies_per_name = Counter(n for names in candidates for n in names)
generic_names = {n for n, c in movies_per_name.items() if c > MAX_NAME_MOVIE_SHARE * len(movies)}
movies["names"] = candidates.apply(lambda names: sorted(names - generic_names))
ALL_NAMES = {n.lower() for names in movies["names"] for n in names}
NAME_PATTERNS = {mid: re.compile(r"\b(?:" + "|".join(re.escape(n) for n in sorted(names, key=len, reverse=True)) + r")\b", re.I)
                 for mid, names in zip(movies["movie_id"], movies["names"]) if names}


def is_name(term):
    return any(tok in ALL_NAMES for tok in term.split())


def mask_names(df, replacement):
    masked = df["review_text"].copy()
    for movie_id, idx in df.groupby("movie_id").groups.items():
        pattern = NAME_PATTERNS.get(movie_id)
        if pattern is not None:
            masked.loc[idx] = masked.loc[idx].str.replace(pattern, replacement, regex=True)
    return masked


print(f"Filme mit Namen: {de(len(NAME_PATTERNS))} von {de(len(movies))} · Namen insgesamt: {de(len(ALL_NAMES))} · "
      f"als allgemein verworfen: {de(len(generic_names))}")
top3 = reviews["movie_id"].value_counts().head(3).index
minfo = movies.set_index("movie_id")
pd.DataFrame({"Anzahl Namen": [len(minfo.at[m, "names"]) for m in top3],
              "Beispiele": [", ".join(minfo.at[m, "names"][:14]) for m in top3]}, index=top3).rename_axis("Film")

## 3 · Datenverständnis (EDA)

Alle Auswertungen in diesem Kapitel nutzen den **vollständigen bereinigten Datensatz**. Sie dienen dem Verständnis, nicht der Modellauswahl, und fließen nicht in Merkmale ein.

### 3.1 Kennzahlen und fehlende Werte

In [ ]:
overview = pd.DataFrame({"Wert": [de(len(reviews)), de(reviews["movie_id"].nunique()), de(reviews["user_id"].nunique()),
                                  de(reviews["label"].sum()), f"{reviews['label'].mean() * 100:.1f} %",
                                  f"{reviews['review_date'].min():%Y-%m-%d} bis {reviews['review_date'].max():%Y-%m-%d}"]},
                        index=["Rezensionen", "Filme", "Nutzer", "Spoiler-Rezensionen", "Spoiler-Anteil", "Zeitraum"]).rename_axis("Kennzahl")
display(save_table(overview, "03_kennzahlen"))

missing = pd.DataFrame({
    "Fehlend (Rezensionen)": reviews[["review_text", "rating", "review_date", "genre"]].isna().sum(),
}).rename_axis("Spalte")
missing.loc["genre", "Fehlend (Rezensionen)"] = int((reviews["genre"].str.len() == 0).sum())
missing_movies = pd.Series({"plot_summary leer": int((movies["plot_summary"].str.len() == 0).sum()),
                            "plot_synopsis leer": int((movies["plot_synopsis"].str.len() == 0).sum()),
                            "keine Figurennamen gefunden": int(len(movies) - len(NAME_PATTERNS))}, name="Fehlend (Filme)")
display(Markdown("**Fehlende Werte**"))
display(missing)
display(missing_movies.to_frame())
note(f"Fehlende Werte betreffen nur Nebenspalten (Bewertung: {de(missing.at['rating', 'Fehlend (Rezensionen)'])}, Datum: "
     f"{de(missing.at['review_date', 'Fehlend (Rezensionen)'])}); Text und Label sind vollständig. "
     f"{de(missing_movies['plot_synopsis leer'])} Filme haben keine ausführliche Handlung; für sie wird in Kap. 8.4 nur die Kurzfassung genutzt. "
     f"Es wird nichts imputiert, weil keine der betroffenen Spalten ein Modellmerkmal ist.")

### 3.2 Klassenverteilung und Zeitverlauf

In [ ]:
by_year = reviews.dropna(subset=["year"]).groupby("year")["label"].agg(["size", "mean"])
by_year = by_year[by_year["size"] >= 100]
counts = reviews["label"].value_counts().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
bars = axes[0].bar(["Kein Spoiler", "Spoiler"], counts.values, color=[LIGHT, DARK])
axes[0].bar_label(bars, labels=[f"{de(v)}\n({v / len(reviews) * 100:.1f} %)" for v in counts.values], fontsize=9)
axes[0].set(title="Klassenverteilung", ylabel="Anzahl Rezensionen", ylim=(0, counts.max() * 1.2))
axes[1].bar(by_year.index, by_year["size"], color=LIGHT)
axes[1].set(title="Rezensionen pro Jahr", xlabel="Jahr", ylabel="Anzahl")
for ax in axes[1:]:
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
axes[2].plot(by_year.index, by_year["mean"] * 100, marker="o", color=DARK, label="Spoiler-Anteil")
axes[2].axhline(reviews["label"].mean() * 100, color=ACCENT, ls="--", label="Gesamtanteil")
axes[2].set(title="Spoiler-Anteil pro Jahr", xlabel="Jahr", ylabel="Spoiler-Anteil in %", ylim=(0, by_year["mean"].max() * 120))
axes[2].legend()
save_fig(fig, "03_klassen_zeit")

imbalance = counts[0] / counts[1]
note(f"Die Klassen sind unausgewogen (Verhältnis {de(imbalance, 1)} : 1). Accuracy wäre deshalb irreführend: „Immer Nicht-Spoiler“ "
     f"erreicht {counts[0] / len(reviews) * 100:.1f} % Accuracy ohne jede Erkennung. Daher werden Precision, Recall, F1, AP und ROC-AUC "
     f"für die Klasse *Spoiler* berichtet (Kap. 4.3). Der Spoiler-Anteil schwankt zwischen den Jahren von "
     f"{by_year['mean'].min() * 100:.0f} % bis {by_year['mean'].max() * 100:.0f} %, was auf wechselndes Markierungsverhalten "
     f"der Verfasser hindeutet (Label-Qualität, Kap. 8.5).")

### 3.3 Textlänge und Ausreißer

In [ ]:
len_stats = reviews.groupby("label")["n_words"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
len_stats.columns = ["Kein Spoiler", "Spoiler"]
display(save_table(len_stats, "03_laenge_statistik"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bins = np.logspace(np.log10(max(1, reviews["n_words"].min())), np.log10(reviews["n_words"].max()), 60)
for label, name, color in [(0, "Kein Spoiler", LIGHT), (1, "Spoiler", DARK)]:
    axes[0].hist(reviews.loc[reviews["label"] == label, "n_words"], bins=bins, density=True, alpha=0.65, color=color, label=name)
axes[0].set(xscale="log", title="Länge der Rezensionen (log-Skala)", xlabel="Wörter", ylabel="Dichte")
axes[0].legend()
axes[1].boxplot([reviews.loc[reviews["label"] == 0, "n_words"], reviews.loc[reviews["label"] == 1, "n_words"]],
                showfliers=True, flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[1].set_xticks([1, 2], ["Kein Spoiler", "Spoiler"])
axes[1].set(yscale="log", title="Länge je Klasse mit Ausreißern", ylabel="Wörter (log)")
save_fig(fig, "03_laenge")

q99 = reviews["n_words"].quantile(0.99)
mw = stats.mannwhitneyu(reviews.loc[reviews["label"] == 1, "n_words"], reviews.loc[reviews["label"] == 0, "n_words"], alternative="greater")
longer = mw.pvalue < 0.05 and len_stats.at['50%', 'Spoiler'] > len_stats.at['50%', 'Kein Spoiler']
note((f"Spoiler-Rezensionen sind deutlich länger (Median {len_stats.at['50%', 'Spoiler']:.0f} vs. {len_stats.at['50%', 'Kein Spoiler']:.0f} Wörter; "
      f"einseitiger Mann-Whitney-U-Test, p = {mw.pvalue:.1e}). Länge ist damit ein **Störfaktor**: Längere Texte enthalten mehr Handlungswörter, "
      f"und ein Modell könnte Länge statt Wortwahl lernen. Kap. 6.6 prüft das explizit. " if longer else
      f"Die Länge unterscheidet sich zwischen den Klassen nicht nennenswert (Median {len_stats.at['50%', 'Spoiler']:.0f} vs. "
      f"{len_stats.at['50%', 'Kein Spoiler']:.0f} Wörter; Mann-Whitney-U, p = {mw.pvalue:.2f}). Kap. 6.6 prüft trotzdem, ob Länge das Modell beeinflusst. ") +
     f"Ausreißer: 1 % der Rezensionen hat mehr als {q99:.0f} Wörter (Maximum {reviews['n_words'].max():,}). Sie werden **nicht entfernt**, "
     f"weil sie echte Rezensionen sind; TF-IDF mit `sublinear_tf` und L2-Normierung dämpft den Längeneffekt auf die Merkmale.")

### 3.4 Bewertung, Filme und Genres

In [ ]:
by_rating = reviews.dropna(subset=["rating"]).groupby("rating")["label"].agg(["size", "mean"])
per_movie = reviews.groupby("movie_id")["label"].agg(["size", "mean"])
genre_stats = (reviews.explode("genre").dropna(subset=["genre"]).groupby("genre")["label"].agg(["size", "mean"])
               .sort_values("size", ascending=False).head(14))

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
ax = axes[0, 0]
ax.bar(by_rating.index, by_rating["mean"] * 100, color=DARK)
ax.axhline(reviews["label"].mean() * 100, color=ACCENT, ls="--", label="Gesamtanteil")
ax.set(title="Spoiler-Anteil nach Bewertung (1–10 Sterne)", xlabel="Bewertung", ylabel="Spoiler-Anteil in %")
ax.legend()
ax = axes[0, 1]
ax.hist(per_movie["size"], bins=60, color=DARK)
ax.set(yscale="log", title="Rezensionen pro Film", xlabel="Anzahl Rezensionen", ylabel="Anzahl Filme (log)")
ax = axes[1, 0]
ax.hist(per_movie.loc[per_movie["size"] >= 20, "mean"] * 100, bins=40, color=DARK)
ax.axvline(reviews["label"].mean() * 100, color=ACCENT, ls="--", label="Gesamtanteil")
ax.set(title="Spoiler-Anteil pro Film (Filme mit ≥ 20 Rezensionen)", xlabel="Spoiler-Anteil in %", ylabel="Anzahl Filme")
ax.legend()
ax = axes[1, 1]
gs_sorted = genre_stats.sort_values("mean")
ax.barh(gs_sorted.index, gs_sorted["mean"] * 100, color=DARK)
ax.axvline(reviews["label"].mean() * 100, color=ACCENT, ls="--")
for i, (g, row) in enumerate(gs_sorted.iterrows()):
    ax.text(row["mean"] * 100 + 0.3, i, f"n={de(row['size'])}", va="center", fontsize=8, color="#555")
ax.set(title="Spoiler-Anteil nach Genre (14 häufigste)", xlabel="Spoiler-Anteil in %", xlim=(0, gs_sorted["mean"].max() * 125))
save_fig(fig, "03_rating_filme_genres")

top10_share = per_movie["size"].sort_values(ascending=False).head(10).sum() / len(reviews)
film_sd = per_movie.loc[per_movie["size"] >= 20, "mean"].std()
note(f"**Bewertung:** Der Spoiler-Anteil variiert mit der Sternezahl (min {by_rating['mean'].min() * 100:.0f} %, max {by_rating['mean'].max() * 100:.0f} %); "
     f"die Bewertung wird bewusst *nicht* als Merkmal verwendet, weil die Forschungsfrage nur die Wortwahl zulässt. "
     f"**Filme:** Die Verteilung ist schief, die 10 meistrezensierten Filme stellen {top10_share * 100:.1f} % aller Rezensionen. "
     f"**Spoiler-Anteil pro Film** streut stark (SD {film_sd * 100:.0f} Prozentpunkte): Der Film allein trägt Information über das Label. "
     f"Bei zufälliger Aufteilung kann ein Modell das über filmspezifische Wörter ausnutzen, bei filmweiser Aufteilung nicht. Genau das testet H2. "
     f"**Genres:** Der Spoiler-Anteil reicht von {gs_sorted['mean'].iloc[0] * 100:.0f} % ({gs_sorted.index[0]}) bis "
     f"{gs_sorted['mean'].iloc[-1] * 100:.0f} % ({gs_sorted.index[-1]}); Kap. 8.2 prüft, ob die Erkennung genreabhängig ist.")

### 3.5 Zusammenhänge zwischen numerischen Größen

Spearman-Korrelationen (robust gegen Ausreißer und schiefe Verteilungen) zwischen Label, Länge, Bewertung, Jahr, Anzahl genannter Figurennamen und der Zahl der Rezensionen des jeweiligen Films.

In [ ]:
reviews["n_names"] = [len(NAME_PATTERNS[m].findall(t)) if m in NAME_PATTERNS else 0
                      for m, t in zip(reviews["movie_id"], reviews["review_text"])]
reviews["movie_n_reviews"] = reviews["movie_id"].map(per_movie["size"])
num = reviews[["label", "n_words", "rating", "year", "n_names", "movie_n_reviews"]].rename(columns={
    "label": "Spoiler", "n_words": "Wörter", "rating": "Bewertung", "year": "Jahr", "n_names": "Figurennamen", "movie_n_reviews": "Rezensionen des Films"})
corr = num.corr(method="spearman")
save_table(corr, "03_korrelationen")

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=35, ha="right")
ax.set_yticks(range(len(corr)), corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iat[i, j]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if abs(corr.iat[i, j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, label="Spearman-Korrelation")
ax.set_title("Zusammenhänge (Spearman)")
save_fig(fig, "03_korrelationen")

note(f"Das Label korreliert am stärksten mit der Länge (ρ = {corr.at['Spoiler', 'Wörter']:.2f}) und mit der Zahl genannter Figurennamen "
     f"(ρ = {corr.at['Spoiler', 'Figurennamen']:.2f}); Länge und Figurennamen hängen selbst zusammen (ρ = {corr.at['Wörter', 'Figurennamen']:.2f}). "
     f"Die stärkste übrige Korrelation mit dem Label beträgt |ρ| = {corr['Spoiler'].drop(['Spoiler', 'Wörter', 'Figurennamen']).abs().max():.2f}. "
     f"Keine numerische Größe ist stark genug, um das Label allein zu erklären; Länge und Namensnennung müssen aber als Störfaktoren mitgeführt werden (Kap. 6.6, 8.1).")

## 4 · Methodik

### 4.1 Stichprobe und Aufteilungen

**Warum eine Stichprobe?** 100.000 Rezensionen halten die Kreuzvalidierung unter 90 Minuten; die Standardfehler der Metriken liegen bei dieser Größe unter 0,01 (Kap. 6.3). Die Stichprobe ist zufällig und reproduzierbar.

**Zwei Aufteilungen, weil die Forschungsfrage zwei Teile hat:**

| Aufteilung | Verfahren | Beantwortet |
|---|---|---|
| **Zufällig** | `train_test_split`, stratifiziert nach Label | Teil (a): bekannte Filme |
| **Nach Film** | `GroupShuffleSplit` mit `movie_id` als Gruppe: kein Film in zwei Mengen | Teil (b): unbekannte Filme |

**Kein zeitbasierter Split**, weil die Rezensionen keine Zeitreihe sind und die Frage nicht auf zukünftige Rezensionen zielt; der Zeitverlauf wird aber als Störquelle der Labels untersucht (Kap. 8.5).

Jede Aufteilung hat drei Teile: **Training** (70 %, darin Kreuzvalidierung für Hyperparameter), **Validierung** (15 %, Schwellenwahl und Modellauswahl), **Test** (15 %, nur Endbewertung).

In [ ]:
sample = reviews.sample(n=min(SAMPLE_SIZE, len(reviews)), random_state=RANDOM_STATE).reset_index(drop=True)
sample["mentions_name"] = sample["n_names"] > 0
sample["text_names_removed"] = mask_names(sample, " ")
sample["text_names_placeholder"] = mask_names(sample, " CHARNAME ")


def split_random(df, seed=RANDOM_STATE):
    train, rest = train_test_split(df, test_size=VAL_SIZE + TEST_SIZE, stratify=df["label"], random_state=seed)
    val, test = train_test_split(rest, test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE), stratify=rest["label"], random_state=seed)
    return {"train": train.copy(), "val": val.copy(), "test": test.copy()}


def split_by_group(df, group_col, seed=RANDOM_STATE):
    outer = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE + TEST_SIZE, random_state=seed)
    tr, rest_idx = next(outer.split(df, groups=df[group_col]))
    train, rest = df.iloc[tr], df.iloc[rest_idx]
    inner = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE), random_state=seed)
    va, te = next(inner.split(rest, groups=rest[group_col]))
    return {"train": train.copy(), "val": rest.iloc[va].copy(), "test": rest.iloc[te].copy()}


SPLITS = {"Zufällig": split_random(sample), "Nach Film": split_by_group(sample, "movie_id")}

rows = []
for split_name, parts in SPLITS.items():
    train_movies, train_users = set(parts["train"]["movie_id"]), set(parts["train"]["user_id"])
    for part, name in [("train", "Training"), ("val", "Validierung"), ("test", "Test")]:
        df = parts[part]
        rows.append({"Aufteilung": split_name, "Teil": name, "Rezensionen": len(df), "Filme": df["movie_id"].nunique(),
                     "Spoiler-Anteil": df["label"].mean(),
                     "Filme auch im Training": "–" if part == "train" else int(df["movie_id"].isin(train_movies).any() and df.loc[df["movie_id"].isin(train_movies), "movie_id"].nunique()),
                     "Anteil Rezensionen von Trainings-Nutzern": np.nan if part == "train" else df["user_id"].isin(train_users).mean()})
split_table = pd.DataFrame(rows).set_index(["Aufteilung", "Teil"])
display(save_table(split_table, "04_aufteilungen"))
user_overlap = split_table.loc[("Nach Film", "Test"), "Anteil Rezensionen von Trainings-Nutzern"]
note(f"Bei der Aufteilung nach Film kommen 0 Testfilme im Training vor; die Spoiler-Anteile der Teile liegen nah beieinander "
     f"(eine Stratifizierung ist bei Gruppen-Splits nicht exakt möglich, der Unterschied ist aber klein). "
     f"**Nutzerüberlappung:** {user_overlap * 100:.0f} % der Testrezensionen (nach Film) stammen von Nutzern, die auch im Training vorkommen. "
     f"Ob das die Ergebnisse verzerrt, prüft die Sensitivitätsanalyse mit Aufteilung nach Nutzer (Kap. 8.6).")

### 4.2 Pipeline

Alle lernenden Schritte stehen in einer `sklearn.Pipeline`, die ausschließlich auf der jeweiligen Trainingsfalte gefittet wird. Damit ist Leakage über Vokabular, IDF-Gewichte oder SVD-Komponenten ausgeschlossen.

| Schritt | Entscheidung | Warum |
|---|---|---|
| Vorverarbeitung | Kleinschreibung, nur Buchstabenwörter ≥ 2 Zeichen; **Spoiler-Warnmarker entfernt** (`spoil…` samt typischer Umgebung: „spoiler alert“, „this review contains spoilers“, „no spoilers“ usw.) | Warnmarker sind Selbstdeklaration des Labels, keine Wortwahl über die Handlung (Fehleranalyse Nr. 1). Ablation in Kap. 8.5. |
| Merkmale | TF-IDF, Unigramme + Bigramme, `min_df=5`, `max_df=0.9`, max. 100.000 Merkmale, `sublinear_tf`, L2-Norm | Bigramme fangen „turns out“, „the end“; sublinear + L2 dämpfen den Längeneffekt. BoW als Ablation (Kap. 8.5). |
| Baum-Ensemble | TF-IDF → `TruncatedSVD` (LSA) → `HistGradientBoostingClassifier` | Bäume auf 100.000 dünn besetzten Dimensionen sind ineffizient und schwach; LSA liefert dichte, informationsreiche Eingaben. |
| Klassenungleichgewicht | `class_weight="balanced"` (LogReg, SVM, Boosting); Complement NB ist für unausgewogene Texte entworfen | Kein Resampling nötig; Schwellenwahl übernimmt die Feinabstimmung. |

In [ ]:
SPOILER_MARKER = re.compile(
    r"(?:\b(?:this\s+review\s+)?(?:contains?|has|with|without|no|any|some|major|minor|possible|slight)\s+)?"
    r"\bspoil\w*(?:\s+(?:alert|warning|ahead|below|free|here))?", re.I)


def preprocess(text):
    # wird innerhalb des Vectorizers angewendet -> Teil der Pipeline
    return SPOILER_MARKER.sub(" ", text.lower())


def make_tfidf(**overrides):
    return TfidfVectorizer(preprocessor=preprocess, sublinear_tf=True, **{**VECTORIZER_ARGS, **overrides})


def make_pipeline(model, with_svd=False, vectorizer=None):
    steps = [("tfidf", vectorizer if vectorizer is not None else make_tfidf())]
    if with_svd:
        steps.append(("svd", TruncatedSVD(n_components=SVD_DIM, random_state=RANDOM_STATE)))
    steps.append(("clf", clone(model)))
    return Pipeline(steps, memory=MEMORY)


MODEL_SPECS = {
    "Naive Bayes": dict(model=ComplementNB(), grid={"clf__alpha": [0.1, 0.3, 1.0]}, svd=False),
    "Log. Regression": dict(model=LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=2000),
                            grid={"clf__C": [0.3, 1.0, 3.0]}, svd=False),
    "Lineare SVM": dict(model=LinearSVC(class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE),
                        grid={"clf__C": [0.03, 0.1, 0.3]}, svd=False),
}
print("Beispiel Vorverarbeitung:", repr(preprocess("SPOILER ALERT! In the end he dies. Spoilers ahead, no spoilers here.")))

### 4.3 Metriken und Validierung

| Metrik | Warum |
|---|---|
| **F1** (Klasse Spoiler) | Harmonisches Mittel aus Precision und Recall; die Hauptmetrik der Präsentation. Hängt von der Schwelle ab. |
| **Average Precision (AP)** | Fläche unter der Precision-Recall-Kurve; schwellenfrei, sensibel für die Minderheitsklasse. Zufall ≈ Spoiler-Anteil (≈ 0,26). **Auswahlkriterium in der Kreuzvalidierung.** |
| **ROC-AUC** | Schwellenfrei, Zufall = 0,5; ergänzend, weil sie bei Ungleichgewicht optimistischer wirkt als AP. |
| Precision, Recall | Zur Interpretation der Fehlerarten. |
| *Accuracy* | **nicht** berichtet (Kap. 3.2). |

**Validierungsschema:** `GridSearchCV` mit `N_FOLDS` Falten auf dem Training (stratifiziert bei zufälliger, `GroupKFold` nach Film bei filmweiser Aufteilung, damit auch innerhalb der CV kein Film in zwei Falten liegt). Das beste Modell je Familie wird auf dem ganzen Training neu gefittet, die **Entscheidungsschwelle auf der Validierungsmenge** F1-optimal gewählt, und erst dann auf der Testmenge bewertet.

**Baselines:** „Immer Spoiler“ (F1-Obergrenze ohne Lernen), „Zufällig raten“ (stratifiziert), „Nur Spoiler-Anteil des Films“ (nutzt nur die Film-ID, kein Text), „Nur Textlänge“ (Kap. 6.6).

In [ ]:
SCORING = {"AP": "average_precision", "F1": "f1", "ROC_AUC": "roc_auc"}


def cv_splitter(split_name):
    if split_name == "Zufällig":
        return StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    return GroupKFold(n_splits=N_FOLDS)


def fit_with_grid(split_name, parts, spec, text_col="review_text"):
    pipe = make_pipeline(spec["model"], with_svd=spec["svd"])
    gs = GridSearchCV(pipe, spec["grid"], scoring=SCORING, refit="AP", cv=cv_splitter(split_name), n_jobs=N_JOBS)
    groups = parts["train"]["movie_id"] if split_name != "Zufällig" else None
    gs.fit(parts["train"][text_col], parts["train"]["label"], groups=groups)
    return gs


def evaluate_on_test(estimator, parts, text_col="review_text"):
    s_val, s_test = score_of(estimator, parts["val"][text_col]), score_of(estimator, parts["test"][text_col])
    thr = best_threshold(parts["val"]["label"], s_val)
    pred = (s_test >= thr).astype(int)
    return {"threshold": thr, "scores": s_test, "preds": pred, **metrics(parts["test"]["label"], pred, s_test)}


def baseline_results(parts):
    y_tr, y_val, y_te = (parts[p]["label"].to_numpy() for p in ("train", "val", "test"))
    out = {}
    out["Immer 'Spoiler'"] = {"scores": np.ones(len(y_te)), "preds": np.ones(len(y_te), dtype=int)}
    rng = np.random.default_rng(RANDOM_STATE)
    rand = rng.random(len(y_te))
    out["Zufällig raten"] = {"scores": rand, "preds": (rand < y_tr.mean()).astype(int)}
    movie_rate = parts["train"].groupby("movie_id")["label"].mean()
    s_val = parts["val"]["movie_id"].map(movie_rate).fillna(y_tr.mean()).to_numpy()
    s_te = parts["test"]["movie_id"].map(movie_rate).fillna(y_tr.mean()).to_numpy()
    out["Nur Spoiler-Anteil des Films"] = {"scores": s_te, "preds": (s_te >= best_threshold(y_val, s_val)).astype(int)}
    for name, d in out.items():
        d.update(metrics(y_te, d["preds"], d["scores"]))
    return out

## 5 · Modellvergleich ⏳

Vier Modellfamilien plus Baselines auf beiden Aufteilungen. Hyperparameter werden per Kreuzvalidierung nach AP gewählt. **Laufzeit: der längste Block des Notebooks** (etwa 30 bis 45 Minuten im vollständigen Modus).

| Modell | Warum im Vergleich |
|---|---|
| Complement Naive Bayes | Klassische Text-Baseline, extrem schnell, robust bei Ungleichgewicht |
| Logistische Regression | Lineares Modell mit direkt interpretierbaren Gewichten (H1) |
| Lineare SVM | Oft bestes lineares Modell für TF-IDF |
| Gradient Boosting auf SVD | Nichtlineares Baum-Ensemble; prüft, ob Interaktionen über lineare Modelle hinaus helfen |

In [ ]:
t0 = time.time()
grids, results, test_preds = {}, [], {}
for split_name, parts in SPLITS.items():
    test_preds[split_name] = {}
    for model_name, spec in MODEL_SPECS.items():
        t1 = time.time()
        gs = fit_with_grid(split_name, parts, spec)
        grids[(split_name, model_name)] = gs
        ev = evaluate_on_test(gs.best_estimator_, parts)
        test_preds[split_name][model_name] = ev
        r = gs.cv_results_
        b = gs.best_index_
        results.append({"Aufteilung": split_name, "Modell": model_name,
                        "Parameter": ", ".join(f"{k.split('__')[1]}={v}" for k, v in gs.best_params_.items()),
                        "CV AP": r["mean_test_AP"][b], "CV AP ±": r["std_test_AP"][b],
                        "CV F1": r["mean_test_F1"][b], "CV F1 ±": r["std_test_F1"][b],
                        "CV ROC-AUC": r["mean_test_ROC_AUC"][b], "CV ROC-AUC ±": r["std_test_ROC_AUC"][b],
                        "Schwelle": ev["threshold"], "Test Precision": ev["Precision"], "Test Recall": ev["Recall"],
                        "Test F1": ev["F1"], "Test AP": ev["AP"], "Test ROC-AUC": ev["ROC-AUC"], "Zeit (s)": time.time() - t1})
        print(f"{split_name:9s} | {model_name:24s} | {results[-1]['Parameter']:18s} | CV-AP {results[-1]['CV AP']:.3f} ± {results[-1]['CV AP ±']:.3f} "
              f"| Test F1 {ev['F1']:.3f} | {time.time() - t1:.0f} s")
    for name, d in baseline_results(parts).items():
        test_preds[split_name][name] = d
        results.append({"Aufteilung": split_name, "Modell": name, "Parameter": "–", "Test Precision": d["Precision"],
                        "Test Recall": d["Recall"], "Test F1": d["F1"], "Test AP": d["AP"], "Test ROC-AUC": d["ROC-AUC"]})
print(f"Gesamt: {(time.time() - t0) / 60:.1f} Minuten")
results = pd.DataFrame(results)

### 5.1 Vergleichstabelle

CV-Werte: Mittelwert ± Standardabweichung über die Falten (nur Trainingsdaten). Test-Werte: einmalige Bewertung auf der Testmenge mit der auf der Validierung gewählten Schwelle. Die Standardabweichung der CV beschreibt die Streuung zwischen Falten; Konfidenzintervalle auf der Testmenge folgen in Kap. 6.3.

In [ ]:
def pm(df, col):
    return [f"{m:.3f} ± {s:.3f}" if pd.notna(s) else "–" for m, s in zip(df[col], df[col + " ±"])]


table = results.copy()
for col in ("CV AP", "CV F1", "CV ROC-AUC"):
    table[col] = pm(table, col)
    table = table.drop(columns=col + " ±")
table = table.drop(columns="Zeit (s)").set_index(["Aufteilung", "Modell"])
save_table(results.set_index(["Aufteilung", "Modell"]), "05_modellvergleich_roh")
display(save_table(table, "05_modellvergleich"))

model_rows = results[results["Modell"].isin(MODEL_SPECS)]
best_name = model_rows[model_rows["Aufteilung"] == "Nach Film"].sort_values("CV AP", ascending=False)["Modell"].iloc[0]
BEST = {s: grids[(s, best_name)].best_estimator_ for s in SPLITS}
always_f1 = results.query("Aufteilung == 'Nach Film' and Modell == \"Immer 'Spoiler'\"")["Test F1"].iloc[0]
best_row = model_rows.query("Aufteilung == 'Nach Film' and Modell == @best_name").iloc[0]
note(f"Bestes Modell nach CV-AP auf der filmweisen Aufteilung (Auswahl ohne Blick auf den Test): **{best_name}** "
     f"(CV-AP {best_row['CV AP']:.3f} ± {best_row['CV AP ±']:.3f}, Test-F1 {best_row['Test F1']:.3f}, Test-AP {best_row['Test AP']:.3f}). "
     f"Die Baseline „Immer Spoiler“ erreicht F1 = {always_f1:.3f}; {'alle' if (model_rows['Test F1'] > always_f1).all() else 'nicht alle'} lernenden Modelle liegen darüber. "
     f"Ob der Abstand signifikant ist, zeigt Kap. 6.3. "
     f"Die CV-Standardabweichungen sind klein (≤ {model_rows['CV AP ±'].max():.3f}), die Rangfolge der Modelle also stabil.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), sharey=True)
for ax, metric in zip(axes, ["AP", "F1"]):
    x = np.arange(len(MODEL_SPECS))
    width = 0.38
    for i, (split_name, color) in enumerate([("Zufällig", LIGHT), ("Nach Film", DARK)]):
        sub = model_rows[model_rows["Aufteilung"] == split_name].set_index("Modell").loc[list(MODEL_SPECS)]
        bars = ax.bar(x + (i - 0.5) * width, sub[f"CV {metric}"], width, yerr=sub[f"CV {metric} ±"], capsize=3, color=color,
                      label=f"{split_name} (CV, Mittel ± SD)")
        ax.scatter(x + (i - 0.5) * width, sub[f"Test {metric}"], color=ACCENT, zorder=3, s=28,
                   label="Testmenge" if i == 0 else None)
    base = results.query("Aufteilung == 'Nach Film' and Modell == \"Immer 'Spoiler'\"")[f"Test {metric}"].iloc[0]
    ax.axhline(base, color=GREY, ls="--", label=f"Baseline „Immer Spoiler“ ({base:.2f})")
    ax.set_xticks(x, [m.replace(" (SVD)", "\n(SVD)") for m in MODEL_SPECS])
    top = max(model_rows[f"CV {metric}"].max(), model_rows[f"Test {metric}"].max())
    ax.set(title=f"{metric}: Kreuzvalidierung und Testmenge", ylabel=metric, ylim=(0, min(1.05, top + 0.25)))
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncols=2, frameon=False)
save_fig(fig, "05_modellvergleich")

## 6 · Evaluation des besten Modells

Alle Analysen in diesem Kapitel beziehen sich auf die **Testmenge** und, wenn nicht anders gesagt, auf die **Aufteilung nach Film** (das ehrliche Szenario der Forschungsfrage).

### 6.1 Konfusionsmatrizen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, split_name in zip(axes, SPLITS):
    ev = test_preds[split_name][best_name]
    ConfusionMatrixDisplay.from_predictions(SPLITS[split_name]["test"]["label"], ev["preds"], display_labels=["Kein Spoiler", "Spoiler"],
                                            cmap="Blues", ax=ax, colorbar=False, values_format="d")
    ax.set(title=f"{split_name} · {best_name}\nF1 = {ev['F1']:.3f} · Precision {ev['Precision']:.2f} · Recall {ev['Recall']:.2f}",
           xlabel="Vorhersage", ylabel="Tatsächlich")
save_fig(fig, "06_konfusionsmatrizen")
ev = test_preds["Nach Film"][best_name]
y_te = SPLITS["Nach Film"]["test"]["label"].to_numpy()
fp, fn = int(((ev["preds"] == 1) & (y_te == 0)).sum()), int(((ev["preds"] == 0) & (y_te == 1)).sum())
note(f"Bei unbekannten Filmen werden {ev['Recall'] * 100:.0f} % der Spoiler gefunden; von allen Alarmen sind {ev['Precision'] * 100:.0f} % berechtigt. "
     f"Das Modell macht {de(fp)} Fehlalarme und übersieht {de(fn)} Spoiler. Die F1-optimale Schwelle erkauft Recall mit Precision; "
     f"für eine Plattform, die Rezensionen vor dem Anzeigen warnt, wäre eher hoher Recall erwünscht, was die Schwellenwahl erlaubt.")

### 6.2 ROC- und Precision-Recall-Kurven

Die PR-Kurve ist bei 26 % Positiven aussagekräftiger als die ROC-Kurve, weil sie den Preis jedes Recall-Gewinns in Fehlalarmen direkt zeigt.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {"Naive Bayes": GREY, "Log. Regression": DARK, "Lineare SVM": LIGHT, "Gradient Boosting (SVD)": GREEN}
y_te = SPLITS["Nach Film"]["test"]["label"].to_numpy()
for model_name in MODEL_SPECS:
    s = test_preds["Nach Film"][model_name]["scores"]
    fpr, tpr, _ = roc_curve(y_te, s)
    pr, rc, _ = precision_recall_curve(y_te, s)
    lw = 2.4 if model_name == best_name else 1.3
    axes[0].plot(fpr, tpr, color=colors[model_name], lw=lw, label=f"{model_name} (AUC {roc_auc_score(y_te, s):.3f})")
    axes[1].plot(rc, pr, color=colors[model_name], lw=lw, label=f"{model_name} (AP {average_precision_score(y_te, s):.3f})")
film_prior = test_preds["Nach Film"]["Nur Spoiler-Anteil des Films"]["scores"]
axes[0].plot([0, 1], [0, 1], ls="--", color=ACCENT, label="Zufall (AUC 0,5)")
axes[1].axhline(y_te.mean(), ls="--", color=ACCENT, label=f"Zufall (AP {y_te.mean():.2f})")
axes[0].set(title="ROC-Kurve (Testmenge, nach Film)", xlabel="Falsch-Positiv-Rate", ylabel="Recall")
axes[1].set(title="Precision-Recall-Kurve (Testmenge, nach Film)", xlabel="Recall", ylabel="Precision", ylim=(0, 1))
for ax in axes:
    ax.legend(fontsize=8)
save_fig(fig, "06_roc_pr")
pr_b, rc_b, _ = precision_recall_curve(y_te, test_preds["Nach Film"][best_name]["scores"])
recall_at_70 = rc_b[pr_b >= 0.7].max() if (pr_b >= 0.7).any() else 0.0
aucs = {m: roc_auc_score(y_te, test_preds["Nach Film"][m]["scores"]) for m in MODEL_SPECS}
note(f"Alle Modelle liegen deutlich über Zufall; die ROC-AUC reicht von {min(aucs.values()):.3f} ({min(aucs, key=aucs.get)}) bis {max(aucs.values()):.3f} ({max(aucs, key=aucs.get)}). "
     f"Die PR-Kurve zeigt den Preis hoher Precision: Bei Precision ≥ 0,7 liegt der Recall von {best_name} bei {recall_at_70:.2f}. "
     f"Ein Teil der Spoiler ist sprachlich nicht von Nicht-Spoilern zu unterscheiden (oder falsch markiert, Kap. 8.5).")

### 6.3 Konfidenzintervalle und Signifikanz gegenüber den Baselines

**Verfahren:** Bootstrap der Testmenge (`N_BOOT` Resamples mit Zurücklegen). Alle Modelle werden auf **denselben** Resamples ausgewertet, dadurch sind Differenzen gepaart und ihre Intervalle enger als bei unabhängigen Vergleichen. Ein Unterschied gilt als signifikant (α = 0,05), wenn das 95-%-Intervall der Differenz die Null nicht enthält; zusätzlich wird der Anteil der Resamples mit Differenz ≤ 0 als empirischer p-Wert berichtet.

In [ ]:
boot = {}
for split_name in SPLITS:
    y = SPLITS[split_name]["test"]["label"].to_numpy()
    boot[split_name] = bootstrap_metrics(y, {name: (d["scores"], d["preds"]) for name, d in test_preds[split_name].items()})

rows = []
for split_name in SPLITS:
    for name, d in boot[split_name].items():
        row = {"Aufteilung": split_name, "Modell": name}
        for m in ("F1", "AP", "ROC-AUC"):
            lo, hi = ci(d[m])
            row[m] = fmt_ci(test_preds[split_name][name][m], lo, hi)
        rows.append(row)
ci_table = pd.DataFrame(rows).set_index(["Aufteilung", "Modell"])
display(Markdown("**Testmetriken mit 95-%-Bootstrap-Konfidenzintervallen**"))
display(save_table(ci_table, "06_konfidenzintervalle"))

rows = []
for split_name in SPLITS:
    for base in ["Immer 'Spoiler'", "Nur Spoiler-Anteil des Films", "Zufällig raten"]:
        for m in ("F1", "AP"):
            diff = boot[split_name][best_name][m] - boot[split_name][base][m]
            lo, hi = ci(diff)
            rows.append({"Aufteilung": split_name, "Vergleich": f"{best_name} − {base}", "Metrik": m,
                         "Differenz": fmt_ci(float(np.mean(diff)), lo, hi), "p (Bootstrap)": float(np.mean(diff <= 0)),
                         "Signifikant": "ja" if lo > 0 else "nein"})
sig_table = pd.DataFrame(rows).set_index(["Aufteilung", "Vergleich", "Metrik"])
display(Markdown("**Ist das beste Modell signifikant besser als die Baselines?**"))
display(save_table(sig_table, "06_signifikanz_baselines"))

d_f1 = boot["Nach Film"][best_name]["F1"] - boot["Nach Film"]["Immer 'Spoiler'"]["F1"]
d_ap = boot["Nach Film"][best_name]["AP"] - boot["Nach Film"]["Nur Spoiler-Anteil des Films"]["AP"]
lo_f1, hi_f1 = ci(d_f1)
lo_ap, hi_ap = ci(d_ap)
findings["Signifikanz vs. Baseline (unbekannte Filme)"] = (
    f"ΔF1 = {np.mean(d_f1):+.3f} [{lo_f1:.3f}; {hi_f1:.3f}] gegen „Immer Spoiler“; ΔAP = {np.mean(d_ap):+.3f} [{lo_ap:.3f}; {hi_ap:.3f}] gegen Film-Prior",
    "signifikant besser" if lo_f1 > 0 and lo_ap > 0 else "nicht durchgängig signifikant")
note(f"Bei unbekannten Filmen ist {best_name} um {np.mean(d_f1):.3f} F1 [{lo_f1:.3f}; {hi_f1:.3f}] besser als „Immer Spoiler“ und um "
     f"{np.mean(d_ap):.3f} AP [{lo_ap:.3f}; {hi_ap:.3f}] besser als die Film-Prior-Baseline. "
     f"{'Beide Intervalle schließen die Null aus: Die Wortwahl trägt signifikant zur Erkennung bei.' if lo_f1 > 0 and lo_ap > 0 else 'Mindestens ein Intervall enthält die Null: Der Vorteil ist nicht gesichert.'} "
     f"Die Intervallbreiten (≈ ±{(hi_f1 - lo_f1) / 2:.3f} F1) geben an, welche Modellunterschiede überhaupt auflösbar sind.")

### 6.4 Unterschiede zwischen den Modellen

Gepaarter Bootstrap-Vergleich jedes Modells mit dem besten Modell (Aufteilung nach Film).

In [ ]:
rows = []
for model_name in MODEL_SPECS:
    if model_name == best_name:
        continue
    for m in ("F1", "AP"):
        diff = boot["Nach Film"][best_name][m] - boot["Nach Film"][model_name][m]
        lo, hi = ci(diff)
        rows.append({"Vergleich": f"{best_name} − {model_name}", "Metrik": m, "Differenz": fmt_ci(float(np.mean(diff)), lo, hi),
                     "Signifikant": "ja" if lo > 0 or hi < 0 else "nein"})
model_diff = pd.DataFrame(rows).set_index(["Vergleich", "Metrik"])
display(save_table(model_diff, "06_modellunterschiede"))
n_sig = (model_diff["Signifikant"] == "ja").sum()
note(f"{n_sig} von {len(model_diff)} Vergleichen sind signifikant. Modelle, deren Intervall die Null enthält, sind statistisch nicht vom besten Modell zu unterscheiden; "
     f"für die Diskussionsfrage „ein Modell oder mehrere?“ heißt das: Die Modellwahl innerhalb der linearen Familie ist zweitrangig, "
     f"die Merkmalsdarstellung (TF-IDF) und die Aufteilung sind wichtiger.")

### 6.5 Merkmalsgewichte

Zur Interpretation dient die **logistische Regression**, auch wenn ein anderes Modell minimal besser ist: Ihre Gewichte sind exakt und ohne Näherung lesbar (für ein lineares Modell liefert SHAP dieselbe Information als Gewicht × Merkmalswert). Positive Gewichte sprechen für Spoiler, negative dagegen. Figurennamen sind orange markiert.

In [ ]:
def coefficients(pipe):
    names = pipe.named_steps["tfidf"].get_feature_names_out()
    return pd.Series(pipe.named_steps["clf"].coef_.ravel(), index=names).sort_values()


LR = {s: grids[(s, "Log. Regression")].best_estimator_ for s in SPLITS}
coef_movie, coef_random = coefficients(LR["Nach Film"]), coefficients(LR["Zufällig"])

fig, axes = plt.subplots(1, 2, figsize=(13, 7.5))
for ax, (title, terms) in zip(axes, [("Stärkste Hinweise auf SPOILER", coef_movie.tail(30)),
                                     ("Stärkste Hinweise auf KEIN Spoiler", coef_movie.head(30)[::-1])]):
    ax.barh(terms.index, terms.values, color=[ACCENT if is_name(t) else DARK for t in terms.index])
    ax.set(title=title, xlabel="Gewicht (logistische Regression, TF-IDF, nach Film)")
    ax.tick_params(axis="y", labelsize=8.5)
save_fig(fig, "06_gewichte")
top_table = pd.DataFrame({"Spoiler-Hinweis": coef_movie.tail(30)[::-1].index, "Gewicht +": coef_movie.tail(30)[::-1].values,
                          "Nicht-Spoiler-Hinweis": coef_movie.head(30).index, "Gewicht −": coef_movie.head(30).values}, index=range(1, 31)).rename_axis("Rang")
save_table(top_table, "06_top_gewichte")

try:
    import shap
    X_tr = LR["Nach Film"].named_steps["tfidf"].transform(SPLITS["Nach Film"]["train"]["review_text"])
    X_te = LR["Nach Film"].named_steps["tfidf"].transform(SPLITS["Nach Film"]["test"]["review_text"][:2000])
    explainer = shap.LinearExplainer(LR["Nach Film"].named_steps["clf"], X_tr, feature_names=coef_movie.sort_index().index)
    shap_values = explainer(X_te)
    shap.plots.bar(shap_values, max_display=20, show=False)
    plt.gcf().set_size_inches(8, 6)
    save_fig(plt.gcf(), "06_shap")
except ImportError:
    print("SHAP ist nicht installiert; für das lineare Modell sind die Koeffizienten die exakte Erklärung (Abbildung oben).")
n_names_top = sum(is_name(t) for t in coef_movie.tail(50).index)
note(f"Stärkste Spoiler-Hinweise: {', '.join(coef_movie.tail(8)[::-1].index)}. Stärkste Nicht-Spoiler-Hinweise: {', '.join(coef_movie.head(8).index)}. "
     f"Unter den 50 stärksten Spoiler-Merkmalen sind {n_names_top} Figurennamen. Das ist der qualitative Befund hinter H1 und H2b; die formalen Tests folgen in Kap. 7.")

### 6.6 Störfaktor Textlänge

Zwei Prüfungen: (1) Wie gut ist eine Baseline, die **nur die Länge** kennt? (2) Bleibt das Modell innerhalb gleich langer Rezensionen besser als die Baseline? Wenn ja, erkennt es mehr als nur „lang“.

In [ ]:
parts = SPLITS["Nach Film"]
len_model = LogisticRegression(class_weight="balanced").fit(np.log1p(parts["train"][["n_words"]]), parts["train"]["label"])
s_val = len_model.decision_function(np.log1p(parts["val"][["n_words"]]))
s_te = len_model.decision_function(np.log1p(parts["test"][["n_words"]]))
pred_len = (s_te >= best_threshold(parts["val"]["label"], s_val)).astype(int)
len_metrics = metrics(parts["test"]["label"], pred_len, s_te)
test_preds["Nach Film"]["Nur Textlänge"] = {"scores": s_te, "preds": pred_len, **len_metrics}

test = parts["test"].copy()
test["pred"], test["score"] = test_preds["Nach Film"][best_name]["preds"], test_preds["Nach Film"][best_name]["scores"]
test["Längenquartil"] = pd.qcut(test["n_words"].rank(method="first"), 4, labels=["Q1 (kurz)", "Q2", "Q3", "Q4 (lang)"])
rows = []
for q, grp in test.groupby("Längenquartil", observed=True):
    p = grp["label"].mean()
    rows.append({"Längenquartil": q, "Wörter": f"{grp['n_words'].min()}–{grp['n_words'].max()}", "Rezensionen": len(grp), "Spoiler-Anteil": p,
                 "F1 Modell": f1_score(grp["label"], grp["pred"], zero_division=0), "AP Modell": average_precision_score(grp["label"], grp["score"]),
                 "F1 „Immer Spoiler“": 2 * p / (1 + p), "AP Zufall": p})
by_len = pd.DataFrame(rows).set_index("Längenquartil")
display(Markdown(f"**Baseline nur aus Textlänge (nach Film):** F1 = {len_metrics['F1']:.3f}, AP = {len_metrics['AP']:.3f}, ROC-AUC = {len_metrics['ROC-AUC']:.3f}"))
display(save_table(by_len, "06_laengenquartile"))

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(by_len))
ax.bar(x - 0.2, by_len["AP Modell"], 0.4, color=DARK, label=f"AP {best_name}")
ax.bar(x + 0.2, by_len["AP Zufall"], 0.4, color=GREY, label="AP Zufall (= Spoiler-Anteil)")
ax.set_xticks(x, by_len.index)
ax.set(title="Modellgüte innerhalb gleich langer Rezensionen (Test, nach Film)", ylabel="Average Precision", ylim=(0, 1))
ax.legend()
save_fig(fig, "06_laenge_quartile")
gain_within = (by_len["AP Modell"] - by_len["AP Zufall"]).min()
findings["Störfaktor Länge"] = (f"Nur-Länge-Baseline AP {len_metrics['AP']:.3f} vs. Modell {test_preds['Nach Film'][best_name]['AP']:.3f}; "
                                f"innerhalb jedes Längenquartils mindestens +{gain_within:.3f} AP über Zufall",
                                "Länge erklärt das Ergebnis nicht" if gain_within > 0.05 else "Länge erklärt einen Großteil des Ergebnisses")
note(f"Die Länge allein erreicht AP = {len_metrics['AP']:.3f}, das Textmodell {test_preds['Nach Film'][best_name]['AP']:.3f}. "
     f"Innerhalb jedes Längenquartils liegt das Modell mindestens {gain_within:.3f} AP über dem Zufallsniveau des Quartils; "
     f"es erkennt also Wortwahl, nicht nur Länge. Der Spoiler-Anteil {'steigt mit der Länge' if by_len['Spoiler-Anteil'].iloc[-1] > by_len['Spoiler-Anteil'].iloc[0] + 0.05 else 'ist über die Quartile ähnlich'} "
     f"({by_len['Spoiler-Anteil'].iloc[0] * 100:.0f} % → {by_len['Spoiler-Anteil'].iloc[-1] * 100:.0f} %), was beim Vergleich absoluter Werte zwischen Quartilen zu beachten ist.")

## 7 · Prüfung der Hypothesen

### 7.1 H1 · Handlung statt Meinung

**Prüfung:** Zwei Wortlisten wurden *vor* der Analyse festgelegt (Grundlage: die Beispiele der Präsentation und allgemeines Filmvokabular). Getestet wird, ob die Modellgewichte der Handlungsliste systematisch höher sind als die der Bewertungsliste (einseitiger Mann-Whitney-U-Test) und welcher Anteil jeder Liste das erwartete Vorzeichen hat. Ergänzend wird der Anteil der Top-50-Spoiler-Merkmale, die zur Handlungsliste gehören, berichtet.

In [ ]:
PLOT_WORDS = ["dies", "die", "died", "death", "dead", "killed", "kills", "kill", "murder", "murdered", "ending", "end", "twist",
              "reveals", "revealed", "reveal", "turns out", "betrays", "escape", "secret", "truth", "finale", "climax", "survives",
              "villain", "sacrifice", "discovers", "married", "pregnant", "returns"]
EVAL_WORDS = ["acting", "performance", "performances", "recommend", "recommended", "brilliant", "boring", "cast", "director",
              "cinematography", "soundtrack", "worth", "entertaining", "enjoyable", "masterpiece", "overrated", "funny", "beautiful",
              "dialogue", "script", "effects", "pacing", "editing", "actors", "actress", "stunning", "disappointing", "excellent",
              "terrible", "waste"]


def lexicon_weights(coef, words):
    return pd.Series({w: coef[w] for w in words if w in coef.index})


plot_w, eval_w = lexicon_weights(coef_movie, PLOT_WORDS), lexicon_weights(coef_movie, EVAL_WORDS)
mw = stats.mannwhitneyu(plot_w, eval_w, alternative="greater")
share_plot_pos, share_eval_neg = (plot_w > 0).mean(), (eval_w < 0).mean()
top50 = coef_movie.tail(50).index
top50_plot = sum(any(tok in PLOT_WORDS for tok in [t] + t.split()) for t in top50)

h1_table = pd.DataFrame({
    "Wörter im Vokabular": [len(plot_w), len(eval_w)],
    "Median-Gewicht": [plot_w.median(), eval_w.median()],
    "Mittelwert": [plot_w.mean(), eval_w.mean()],
    "Anteil mit erwartetem Vorzeichen": [share_plot_pos, share_eval_neg],
}, index=["Handlungswörter (erwartet > 0)", "Bewertungswörter (erwartet < 0)"]).rename_axis("Wortliste")
display(save_table(h1_table, "07_h1_wortlisten"))
print(f"Mann-Whitney-U (Handlung > Bewertung): U = {mw.statistic:.0f}, p = {mw.pvalue:.2e}")
print(f"Top-50-Spoiler-Merkmale aus der Handlungsliste: {top50_plot} von 50")

fig, ax = plt.subplots(figsize=(10, 4.5))
both = pd.concat([plot_w.rename("w").to_frame().assign(Liste="Handlung"), eval_w.rename("w").to_frame().assign(Liste="Bewertung")])
for i, (name, color) in enumerate([("Handlung", DARK), ("Bewertung", LIGHT)]):
    vals = both.loc[both["Liste"] == name, "w"].sort_values()
    ax.scatter(vals.values, np.full(len(vals), i) + np.linspace(-0.25, 0.25, len(vals)), color=color, s=28, label=name)
    extremes = pd.concat([vals.head(2), vals.tail(2)])
    for k, (term, v) in enumerate(extremes.items()):
        ax.annotate(term, (v, i), fontsize=8, xytext=(0, 14 + 10 * (k % 2)), textcoords="offset points", ha="center")
ax.axvline(0, color=ACCENT, ls="--")
ax.set_yticks([0, 1], ["Handlungswörter", "Bewertungswörter"])
ax.set(title="H1: Gewichte der vorab definierten Wortlisten (LogReg, nach Film)", xlabel="Gewicht (> 0 spricht für Spoiler)", ylim=(-0.6, 1.8))
save_fig(fig, "07_h1")

h1_verdict = "belegt" if (mw.pvalue < 0.05 and plot_w.median() > 0 > eval_w.median()) else ("unentschieden" if mw.pvalue < 0.05 else "widerlegt")
findings["H1 · Handlung statt Meinung"] = (
    f"Median-Gewicht Handlung {plot_w.median():+.2f} vs. Bewertung {eval_w.median():+.2f}; {share_plot_pos * 100:.0f} % bzw. {share_eval_neg * 100:.0f} % mit erwartetem Vorzeichen; "
    f"Mann-Whitney p = {mw.pvalue:.1e}; {top50_plot}/50 Top-Merkmale aus der Handlungsliste", h1_verdict)
note(f"**H1 ist {h1_verdict}.** Handlungswörter haben signifikant höhere Gewichte als Bewertungswörter (p = {mw.pvalue:.1e}); "
     f"{share_plot_pos * 100:.0f} % der Handlungswörter sprechen für Spoiler, {share_eval_neg * 100:.0f} % der Bewertungswörter dagegen. "
     f"{'Die Ausnahmen (Abbildung) sind Wörter, die in beiden Kontexten vorkommen, etwa „twist“ als Lob. ' if min(share_plot_pos, share_eval_neg) < 1 else ''}"
     f"Limitation: Die Listen sind klein und von uns gewählt; sie wurden aber vor dem Blick auf die Gewichte fixiert und sind im Code dokumentiert.")

### 7.2 H2 · Unbekannte Filme sind schwerer, weil Figurennamen fehlen ⏳

Die Hypothese hat zwei Teile und wird in zwei Schritten geprüft:

**(a) Effekt:** Ist die Güte bei filmweiser Aufteilung niedriger? `N_SEEDS` Wiederholungen mit jeweils neuer zufälliger *und* filmweiser Aufteilung derselben Stichprobe, logistische Regression mit den in Kap. 5 gewählten Hyperparametern, Schwelle je Durchlauf auf der Validierung. Paarweiser Vergleich (gleicher Seed → gleiche Stichprobe), t-Test für gepaarte Stichproben und Wilcoxon-Test, 95-%-Konfidenzintervall der mittleren Differenz.

**(b) Mechanismus:** Liegt es an Figurennamen? Wenn ja, muss der Abstand zwischen den Aufteilungen schrumpfen, sobald die Namen aus allen Texten entfernt sind (gleiche Seeds, gleiche Modelle, nur Text ohne Namen).

In [ ]:
lr_params = {k.split("__")[1]: v for k, v in grids[("Nach Film", "Log. Regression")].best_params_.items()}


def fit_lr(parts, text_col="review_text", vectorizer=None):
    pipe = make_pipeline(LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=2000, **lr_params), vectorizer=vectorizer)
    pipe.fit(parts["train"][text_col], parts["train"]["label"])
    return pipe, evaluate_on_test(pipe, parts, text_col)


t0 = time.time()
rows = []
for seed in range(N_SEEDS):
    for split_name, fn in [("Zufällig", split_random), ("Nach Film", lambda d, s: split_by_group(d, "movie_id", s))]:
        parts_s = fn(sample, RANDOM_STATE + 1 + seed)
        for variant, col in [("Original", "review_text"), ("Namen entfernt", "text_names_removed")]:
            _, ev = fit_lr(parts_s, col)
            rows.append({"Seed": seed, "Aufteilung": split_name, "Variante": variant, "F1": ev["F1"], "AP": ev["AP"], "ROC-AUC": ev["ROC-AUC"]})
    print(f"Seed {seed + 1}/{N_SEEDS} fertig ({(time.time() - t0) / 60:.1f} min)")
h2 = pd.DataFrame(rows)
h2_summary = h2.groupby(["Variante", "Aufteilung"])[["F1", "AP", "ROC-AUC"]].agg(["mean", "std"])
display(save_table(h2_summary, "07_h2_seeds"))

In [ ]:
def paired_test(variant, metric):
    wide = h2[h2["Variante"] == variant].pivot(index="Seed", columns="Aufteilung", values=metric)
    diff = wide["Nach Film"] - wide["Zufällig"]
    t = stats.ttest_rel(wide["Nach Film"], wide["Zufällig"])
    w = stats.wilcoxon(diff) if len(diff) >= 5 and (diff != 0).any() else None
    half = stats.t.ppf(0.975, len(diff) - 1) * diff.std(ddof=1) / np.sqrt(len(diff))
    return {"Variante": variant, "Metrik": metric, "Δ (Film − Zufällig)": diff.mean(), "95-%-KI": f"[{diff.mean() - half:.3f}; {diff.mean() + half:.3f}]",
            "KI unten": diff.mean() - half, "KI oben": diff.mean() + half, "p (t-Test)": t.pvalue, "p (Wilcoxon)": w.pvalue if w else np.nan,
            "Seeds mit Δ < 0": f"{int((diff < 0).sum())}/{len(diff)}"}


h2_tests = pd.DataFrame([paired_test(v, m) for v in ("Original", "Namen entfernt") for m in ("F1", "AP")]).set_index(["Variante", "Metrik"])
display(save_table(h2_tests.drop(columns=["KI unten", "KI oben"]), "07_h2_tests"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, metric in zip(axes, ["F1", "AP"]):
    for i, variant in enumerate(["Original", "Namen entfernt"]):
        for j, (split_name, color) in enumerate([("Zufällig", LIGHT), ("Nach Film", DARK)]):
            vals = h2[(h2["Variante"] == variant) & (h2["Aufteilung"] == split_name)][metric]
            ax.errorbar(i + (j - 0.5) * 0.25, vals.mean(), yerr=vals.std(ddof=1), fmt="o", color=color, capsize=4, markersize=7,
                        label=split_name if i == 0 else None)
            ax.scatter(np.full(len(vals), i + (j - 0.5) * 0.25) + np.linspace(-0.04, 0.04, len(vals)), vals, color=color, alpha=0.4, s=12)
    ax.set_xticks([0, 1], ["Original", "Namen entfernt"])
    ax.set(title=f"H2: {metric} über {N_SEEDS} Aufteilungen (Mittel ± SD)", ylabel=metric)
    ax.legend()
save_fig(fig, "07_h2")

eff = h2_tests.loc[("Original", "F1")]
eff_nn = h2_tests.loc[("Namen entfernt", "F1")]
part_a = "belegt" if eff["KI oben"] < 0 else ("widerlegt" if eff["KI unten"] > 0 else "unentschieden")
gap_orig, gap_nn = -eff["Δ (Film − Zufällig)"], -eff_nn["Δ (Film − Zufällig)"]
part_b = "nicht prüfbar (kein Vorsprung bekannter Filme)" if gap_orig <= 0.005 else ("widerlegt" if gap_nn >= 0.5 * gap_orig else "belegt")
findings["H2a · Unbekannte Filme sind schwerer"] = (
    f"ΔF1 = {eff['Δ (Film − Zufällig)']:+.3f} {eff['95-%-KI']}, p = {eff['p (t-Test)']:.3f} (t-Test), {eff['Seeds mit Δ < 0']} Seeds negativ", part_a)
findings["H2b · … weil Figurennamen fehlen"] = (
    f"Abstand zufällig − Film: {gap_orig:+.3f} F1 mit Namen, {gap_nn:+.3f} ohne Namen", part_b)
note(f"**H2a ({part_a}).** Bei unbekannten Filmen ist der F1 im Mittel um {abs(eff['Δ (Film − Zufällig)']):.3f} "
     f"{'niedriger' if eff['Δ (Film − Zufällig)'] < 0 else 'höher'} (95-%-KI {eff['95-%-KI']}, p = {eff['p (t-Test)']:.3f}). "
     f"Der Effekt ist {'klein, aber konsistent' if part_a == 'belegt' else 'nicht gesichert'}: Mit {N_SEEDS} Seeds ist der t-Test konservativ, "
     f"die Richtung in {eff['Seeds mit Δ < 0']} Durchläufen dieselbe. "
     f"**H2b ({part_b}).** Ohne Figurennamen beträgt der Abstand {gap_nn:+.3f} statt {gap_orig:+.3f} F1. "
     f"{'Der Abstand verschwindet nicht, wenn Namen fehlen: Figurennamen sind nicht der Mechanismus.' if part_b == 'widerlegt' else 'Der Abstand schrumpft deutlich: Figurennamen tragen zum Vorteil bekannter Filme bei.' if part_b == 'belegt' else 'Ohne messbaren Vorsprung bekannter Filme lässt sich der Mechanismus nicht prüfen.'} "
     f"Kap. 8.1 vertieft das mit einem Platzhalter-Experiment.")

### 7.3 H3 · Wörter wie „killed“ oder „murder“ lösen Fehlalarme aus

**Prüfung:** Unter den *Nicht-Spoilern* der Testmenge (nach Film) wird die Fehlalarmquote des besten Modells mit und ohne Alarmwort verglichen. Weil Rezensionen mit solchen Wörtern länger sind (Störfaktor), wird das Verhältnis **längenadjustiert**: Die Fehlalarmquote ohne Wort wird innerhalb von Längendezilen berechnet und auf die Längenverteilung der Rezensionen *mit* Wort standardisiert. Das Konfidenzintervall kommt aus dem Bootstrap über die Rezensionen.

In [ ]:
ALARM_WORDS = ["killed", "kills", "murder", "murdered", "dies", "death", "dead"]
neg = test[test["label"] == 0].copy()
for w in ALARM_WORDS:
    neg[w] = neg["review_text"].str.contains(rf"\b{w}\b", case=False, regex=True)
neg["killed/murder"] = neg["killed"] | neg["murder"] | neg["kills"] | neg["murdered"]
neg["decile"] = pd.qcut(neg["n_words"].rank(method="first"), 10, labels=False)


def adjusted_ratio(df, word):
    has = df[word]
    if not has.any() or has.all():
        return np.nan, np.nan, np.nan
    raw_with, raw_without = df.loc[has, "pred"].mean(), df.loc[~has, "pred"].mean()
    weights = df.loc[has].groupby("decile").size()
    fpr_without = df.loc[~has].groupby("decile")["pred"].mean()
    expected = (weights * fpr_without.reindex(weights.index).fillna(raw_without)).sum() / weights.sum()
    return raw_with, raw_without, raw_with / expected


rows = []
for w in ALARM_WORDS + ["killed/murder"]:
    raw_with, raw_without, adj = adjusted_ratio(neg, w)
    rows.append({"Wort": w, "Nicht-Spoiler mit Wort": int(neg[w].sum()), "Fehlalarmquote mit Wort": raw_with,
                 "Fehlalarmquote ohne Wort": raw_without, "Verhältnis roh": raw_with / raw_without if raw_without else np.nan,
                 "Verhältnis längenadjustiert": adj})
h3_table = pd.DataFrame(rows).set_index("Wort")
display(save_table(h3_table, "07_h3_fehlalarme"))

rng = np.random.default_rng(RANDOM_STATE)
boot_ratio = []
for _ in range(N_BOOT):
    res = neg.iloc[rng.integers(0, len(neg), len(neg))]
    boot_ratio.append(adjusted_ratio(res, "killed/murder")[2])
boot_ratio = np.array([r for r in boot_ratio if np.isfinite(r)])
lo, hi = ci(boot_ratio)
point = h3_table.at["killed/murder", "Verhältnis längenadjustiert"]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(h3_table))
ax.bar(x - 0.2, h3_table["Fehlalarmquote ohne Wort"] * 100, 0.4, color=LIGHT, label="ohne Wort")
ax.bar(x + 0.2, h3_table["Fehlalarmquote mit Wort"] * 100, 0.4, color=ACCENT, label="mit Wort")
for i, adj in enumerate(h3_table["Verhältnis längenadjustiert"]):
    top = np.nanmax([h3_table["Fehlalarmquote mit Wort"].iloc[i], h3_table["Fehlalarmquote ohne Wort"].iloc[i]])
    ax.text(i, (top if np.isfinite(top) else 0) * 100 + 1, f"×{adj:.1f}" if np.isfinite(adj) else "", ha="center", fontsize=8)
ax.set_xticks(x, h3_table.index)
ax.set(title="H3: Fehlalarmquote bei Nicht-Spoilern (Test, nach Film) · Zahl = längenadjustiertes Verhältnis", ylabel="Fehlalarmquote in %",
       ylim=(0, max(h3_table["Fehlalarmquote mit Wort"].max() * 100 * 1.25, 10)))
ax.legend()
save_fig(fig, "07_h3")

h3_verdict = "belegt" if lo > 1 else ("widerlegt" if hi < 1 else "unentschieden")
findings["H3 · killed/murder lösen Fehlalarme aus"] = (
    f"Fehlalarmquote {h3_table.at['killed/murder', 'Fehlalarmquote mit Wort'] * 100:.0f} % mit vs. {h3_table.at['killed/murder', 'Fehlalarmquote ohne Wort'] * 100:.0f} % ohne; "
    f"längenadjustiertes Verhältnis {point:.2f} [{lo:.2f}; {hi:.2f}]", h3_verdict)
note(f"**H3 ist {h3_verdict}.** Nicht-Spoiler, die „killed“, „kills“, „murder“ oder „murdered“ enthalten, werden {point:.2f}-mal so oft fälschlich als Spoiler markiert "
     f"wie gleich lange Nicht-Spoiler ohne diese Wörter (95-%-KI [{lo:.2f}; {hi:.2f}]). Das rohe Verhältnis ({h3_table.at['killed/murder', 'Verhältnis roh']:.2f}) "
     f"{'ist höher, ein Teil des Effekts geht also auf die Länge zurück' if h3_table.at['killed/murder', 'Verhältnis roh'] > point else 'ist nicht höher, Länge erklärt den Effekt also nicht'}; "
     f"{'nach Adjustierung bleibt ein signifikanter Effekt' if lo > 1 else 'nach Adjustierung ist der Effekt nicht gesichert'}. "
     f"Ein Bag-of-Words-Modell kann „the villain gets killed in the end“ nicht von „nobody gets killed, it's a comedy“ unterscheiden.")

## 8 · Zusatzanalysen

### 8.1 Figurennamen: entfernen oder durch Platzhalter ersetzen

Ergänzung zu H2b auf der Hauptaufteilung. Ein Platzhalter (`CHARNAME`) prüft, ob die bloße Information „hier wird eine Figur genannt“ auf neue Filme übertragbar ist, auch wenn der konkrete Name es nicht ist.

In [ ]:
VARIANTS = {"Original": "review_text", "Namen entfernt": "text_names_removed", "Platzhalter": "text_names_placeholder"}
rows, name_models = [], {}
for split_name, parts in SPLITS.items():
    for variant, col in VARIANTS.items():
        pipe, ev = fit_lr(parts, col)
        name_models[(split_name, variant)] = (pipe, ev)
        rows.append({"Aufteilung": split_name, "Variante": variant, "F1": ev["F1"], "AP": ev["AP"], "ROC-AUC": ev["ROC-AUC"]})
names_res = pd.DataFrame(rows)
names_wide = names_res.pivot(index="Variante", columns="Aufteilung", values="AP").loc[list(VARIANTS), ["Zufällig", "Nach Film"]]
names_wide["Abstand"] = names_wide["Zufällig"] - names_wide["Nach Film"]
display(save_table(names_res.set_index(["Aufteilung", "Variante"]), "08_namen"))

y = SPLITS["Nach Film"]["test"]["label"].to_numpy()
b = bootstrap_metrics(y, {v: (name_models[("Nach Film", v)][1]["scores"], name_models[("Nach Film", v)][1]["preds"]) for v in VARIANTS}, n_boot=min(N_BOOT, 400))
rows = []
for v in ("Namen entfernt", "Platzhalter"):
    for m in ("F1", "AP"):
        d = b[v][m] - b["Original"][m]
        lo, hi = ci(d)
        rows.append({"Variante": v, "Metrik": m, "Δ zu Original (nach Film)": fmt_ci(float(d.mean()), lo, hi), "Signifikant": "ja" if lo > 0 or hi < 0 else "nein"})
display(save_table(pd.DataFrame(rows).set_index(["Variante", "Metrik"]), "08_namen_differenzen"))
coef_ph = coefficients(name_models[("Nach Film", "Platzhalter")][0])
ph_weight = coef_ph.get("charname", np.nan)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(VARIANTS))
for j, (split_name, color) in enumerate([("Zufällig", LIGHT), ("Nach Film", DARK)]):
    bars = ax.bar(x + (j - 0.5) * 0.38, names_wide[split_name], 0.38, color=color, label=split_name)
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)
ax.set_xticks(x, list(VARIANTS))
lo_y = names_wide[["Zufällig", "Nach Film"]].min().min()
ax.set(title="Einfluss der Figurennamen (LogReg, Testmenge)", ylabel="AP", ylim=(max(0, lo_y - 0.06), names_wide[["Zufällig", "Nach Film"]].max().max() + 0.03))
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
save_fig(fig, "08_namen")
findings["Figurennamen (Platzhalter)"] = (
    f"AP nach Film: Original {names_wide.at['Original', 'Nach Film']:.3f}, ohne Namen {names_wide.at['Namen entfernt', 'Nach Film']:.3f}, Platzhalter {names_wide.at['Platzhalter', 'Nach Film']:.3f}; Gewicht CHARNAME {ph_weight:+.2f}",
    "Namen sind kein Treiber; Platzhalter ändert nichts Signifikantes" if all(r["Signifikant"] == "nein" for r in rows) else "Namensbehandlung hat messbaren Effekt")
sig_rows = [f"{r['Variante']} ({r['Metrik']})" for r in rows if r["Signifikant"] == "ja"]
intro = ("Weder das Entfernen noch der Platzhalter verändern die Güte bei unbekannten Filmen signifikant (Tabelle). " if not sig_rows
         else f"Signifikante Änderungen bei unbekannten Filmen: {', '.join(sig_rows)}. ")
note(intro +
     f"Der Platzhalter bekommt ein hohes Gewicht ({ph_weight:+.2f}: Figuren zu nennen ist typisch für Nacherzählung). "
     f"Der Vorsprung bekannter Filme (Abstand {names_wide.at['Original', 'Abstand']:+.3f} AP) beträgt ohne Namen {names_wide.at['Namen entfernt', 'Abstand']:+.3f}; "
     + ("es gibt hier keinen nennenswerten Vorsprung bekannter Filme, den Namen erklären könnten." if names_wide.at['Original', 'Abstand'] < 0.005 else
        "er bleibt also bestehen und stammt vermutlich aus anderen filmspezifischen Wörtern (Orte, Objekte, Ereignisse), nicht aus Namen." if names_wide.at['Namen entfernt', 'Abstand'] >= 0.5 * names_wide.at['Original', 'Abstand'] else
        "er schrumpft also deutlich, Namen tragen zum Vorteil bekannter Filme bei."))

### 8.2 Genres

(1) Modellgüte je Genre mit Bootstrap-Konfidenzintervall (eine Rezension zählt in jedem ihrer Genres). (2) Hilft ein eigenes Modell pro Genre? Vergleich auf den unbekannten Testfilmen des Genres: Genre-Modell vs. allgemeines Modell mit **gleich vielen** Trainingsrezensionen (fairer Vergleich) vs. allgemeines Modell mit allen Daten.

In [ ]:
per_genre = test.explode("genre").dropna(subset=["genre"])
rows = []
for genre, grp in per_genre.groupby("genre"):
    if len(grp) < MIN_GENRE_TEST or grp["label"].sum() < 20 or grp["label"].nunique() < 2:
        continue
    b = bootstrap_metrics(grp["label"].to_numpy(), {"m": (grp["score"].to_numpy(), grp["pred"].to_numpy())}, n_boot=min(N_BOOT, 300))["m"]
    p = grp["label"].mean()
    rows.append({"Genre": genre, "Rezensionen": len(grp), "Filme": grp["movie_id"].nunique(), "Spoiler-Anteil": p,
                 "AP": average_precision_score(grp["label"], grp["score"]), "AP KI unten": ci(b["AP"])[0], "AP KI oben": ci(b["AP"])[1],
                 "F1": f1_score(grp["label"], grp["pred"], zero_division=0), "F1 „Immer Spoiler“": 2 * p / (1 + p)})
genre_eval = pd.DataFrame(rows).set_index("Genre").sort_values("AP")
genre_eval["AP über Zufall"] = genre_eval["AP"] - genre_eval["Spoiler-Anteil"]
display(save_table(genre_eval.sort_values("AP", ascending=False), "08_genres_guete"))

fig, ax = plt.subplots(figsize=(10, 0.4 * len(genre_eval) + 1.5))
ax.errorbar(genre_eval["AP"], genre_eval.index, xerr=[genre_eval["AP"] - genre_eval["AP KI unten"], genre_eval["AP KI oben"] - genre_eval["AP"]],
            fmt="o", color=DARK, capsize=3, label="AP des Modells (95-%-KI)")
ax.scatter(genre_eval["Spoiler-Anteil"], genre_eval.index, color=ACCENT, zorder=3, label="Zufall (= Spoiler-Anteil)")
for i, (g, row) in enumerate(genre_eval.iterrows()):
    ax.text(row["AP KI oben"] + 0.01, i, f"{int(row['Filme'])} Filme", va="center", fontsize=8, color="#555")
ax.set(xlabel="Average Precision (Test, nach Film)", title="Erkennung nach Genre", xlim=(0, genre_eval["AP KI oben"].max() + 0.12))
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
save_fig(fig, "08_genres")
overlap = genre_eval["AP KI unten"].max() <= genre_eval["AP KI oben"].min()
note(f"Die AP reicht von {genre_eval['AP'].min():.3f} ({genre_eval['AP'].idxmin()}) bis {genre_eval['AP'].max():.3f} ({genre_eval['AP'].idxmax()}). "
     f"{'Die Konfidenzintervalle aller Genres überlappen, die Unterschiede sind also nicht gesichert.' if overlap else 'Mindestens zwei Genres unterscheiden sich signifikant (nicht überlappende Intervalle).'} "
     f"Pro Genre liegen nur {int(genre_eval['Filme'].min())} bis {int(genre_eval['Filme'].max())} Filme in der Testmenge; Genre-Aussagen sind deshalb vorsichtig zu formulieren.")

In [ ]:
def in_genre(df, genre):
    return df["genre"].apply(lambda gs: genre in gs)


train_m, test_m = SPLITS["Nach Film"]["train"], SPLITS["Nach Film"]["test"]
top_genres = sample.explode("genre")["genre"].dropna().value_counts().head(N_GENRES).index
rows = []
for genre in top_genres:
    train_g, test_g = train_m[in_genre(train_m, genre)], test_m[in_genre(test_m, genre)]
    if len(test_g) < MIN_GENRE_TEST or test_g["label"].nunique() < 2:
        continue
    same_size = train_m.sample(n=len(train_g), random_state=RANDOM_STATE)
    y_g = test_g["label"].to_numpy()
    variants = {}
    for variant, tr in [("Genre-Modell", train_g), ("Allgemein, gleich groß", same_size)]:
        pipe = make_pipeline(LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=2000, **lr_params)).fit(tr["review_text"], tr["label"])
        variants[variant] = score_of(pipe, test_g["review_text"])
    variants["Allgemein, voll"] = score_of(LR["Nach Film"], test_g["review_text"])
    b = bootstrap_metrics(y_g, {k: (s, (s >= 0).astype(int)) for k, s in variants.items()}, n_boot=min(N_BOOT, 300))
    d = b["Genre-Modell"]["AP"] - b["Allgemein, gleich groß"]["AP"]
    lo, hi = ci(d)
    rows.append({"Genre": genre, "Training (Genre)": len(train_g), "Test": len(test_g),
                 **{f"AP {k}": average_precision_score(y_g, s) for k, s in variants.items()},
                 "Δ Genre − gleich groß": fmt_ci(float(d.mean()), lo, hi), "Signifikant": "ja" if lo > 0 or hi < 0 else "nein"})
    print(f"{genre}: fertig")
genre_models = pd.DataFrame(rows).set_index("Genre")
display(save_table(genre_models, "08_genre_modelle"))
wins = (genre_models["Signifikant"] == "ja").sum()
findings["Genres"] = (f"AP-Spanne {genre_eval['AP'].min():.3f}–{genre_eval['AP'].max():.3f}; Genre-Modell signifikant anders als gleich großes allgemeines Modell in {wins}/{len(genre_models)} Genres",
                      "Spoiler-Sprache ist weitgehend genreübergreifend" if wins == 0 else "Genre-Modelle lohnen sich teilweise")
note(f"Ein eigenes Modell pro Genre ist in {wins} von {len(genre_models)} Genres signifikant anders als ein gleich großes allgemeines Modell; "
     f"das allgemeine Modell mit allen Daten ist durchweg mindestens gleich gut. Spoiler-Sprache ist also weitgehend genreübergreifend, mehr Daten schlagen Spezialisierung.")

### 8.3 Einzelne Filme: Was bringt es, einen Film zu kennen? ⏳

Für die `N_FILMS` meistrezensierten Filme (gesamter Datensatz): 5-fache stratifizierte Kreuzvalidierung innerhalb des Films. In jeder Falte: **Nur dieser Film** (Training nur auf den übrigen Rezensionen des Films), **Allgemein, Film nie gesehen** (Modell aus `FILM_GENERAL_N` Rezensionen anderer Filme), **Allgemein + dieser Film**. Mittelwert ± SD über die Falten, dadurch belastbarer als eine einzelne Aufteilung.

In [ ]:
film_ids = reviews["movie_id"].value_counts().head(N_FILMS).index
display(pd.DataFrame({"Rezensionen": reviews["movie_id"].value_counts().loc[film_ids],
                      "Spoiler-Anteil": reviews.groupby("movie_id")["label"].mean().loc[film_ids],
                      "Genres": [", ".join(minfo.at[m, "genre"]) for m in film_ids],
                      "Handlung (Anfang)": [textwrap.shorten(minfo.at[m, "plot_summary"], 90, placeholder=" …") for m in film_ids]}, index=film_ids).rename_axis("Film"))

t0 = time.time()
rows = []
for movie_id in film_ids:
    film = reviews[reviews["movie_id"] == movie_id].reset_index(drop=True)
    others = sample[sample["movie_id"] != movie_id]
    general = others.sample(n=min(FILM_GENERAL_N, len(others)), random_state=RANDOM_STATE)
    general_pipe = make_pipeline(LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=2000, **lr_params)).fit(general["review_text"], general["label"])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr, te) in enumerate(skf.split(film, film["label"])):
        f_tr, f_te = film.iloc[tr], film.iloc[te]
        for variant, pipe in [
            ("Nur dieser Film", make_pipeline(LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=2000, **lr_params),
                                              vectorizer=make_tfidf(min_df=2)).fit(f_tr["review_text"], f_tr["label"])),
            ("Allgemein (Film nie gesehen)", general_pipe),
            ("Allgemein + dieser Film", make_pipeline(LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=2000, **lr_params))
                                        .fit(pd.concat([general["review_text"], f_tr["review_text"]]), pd.concat([general["label"], f_tr["label"]]))),
        ]:
            s = score_of(pipe, f_te["review_text"])
            rows.append({"Film": movie_id, "Falte": fold, "Variante": variant, "AP": average_precision_score(f_te["label"], s),
                         "ROC-AUC": roc_auc_score(f_te["label"], s)})
    print(f"{movie_id}: fertig ({(time.time() - t0) / 60:.1f} min)")
film_res = pd.DataFrame(rows)
film_summary = film_res.groupby(["Film", "Variante"])[["AP", "ROC-AUC"]].agg(["mean", "std"])
display(save_table(film_summary, "08_einzelfilme"))

In [ ]:
order = ["Nur dieser Film", "Allgemein (Film nie gesehen)", "Allgemein + dieser Film"]
film_ap = film_res.pivot_table(index="Film", columns="Variante", values="AP", aggfunc=["mean", "std"])
fig, ax = plt.subplots(figsize=(10, 4.2))
x = np.arange(len(film_ids))
for i, (variant, color) in enumerate(zip(order, [ACCENT, DARK, LIGHT])):
    m, s = film_ap[("mean", variant)].loc[film_ids], film_ap[("std", variant)].loc[film_ids]
    bars = ax.bar(x + (i - 1) * 0.27, m, 0.27, yerr=s, capsize=3, color=color, label=variant)
    ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=3)
ax.set_xticks(x, film_ids)
ax.set(title="Einzelne Filme: AP über 5 Falten (Mittel ± SD)", ylabel="Average Precision", ylim=(0, film_ap["mean"].max().max() * 1.3))
ax.legend(ncols=3, fontsize=8, loc="upper left")
save_fig(fig, "08_einzelfilme")

paired = film_res.pivot_table(index=["Film", "Falte"], columns="Variante", values="AP")
d_known = paired["Allgemein + dieser Film"] - paired["Allgemein (Film nie gesehen)"]
d_only = paired["Nur dieser Film"] - paired["Allgemein (Film nie gesehen)"]
t_known, t_only = stats.ttest_rel(paired["Allgemein + dieser Film"], paired["Allgemein (Film nie gesehen)"]), stats.ttest_rel(paired["Nur dieser Film"], paired["Allgemein (Film nie gesehen)"])
findings["Einzelne Filme"] = (f"Film im Training: ΔAP = {d_known.mean():+.3f} (p = {t_known.pvalue:.3f}, {len(d_known)} Falten); Nur-Film-Modell vs. allgemein: {d_only.mean():+.3f} (p = {t_only.pvalue:.3f})",
                              "Den Film zu kennen hilft messbar" if (t_known.pvalue < 0.05 and d_known.mean() > 0) else "Den Film zu kennen hilft nicht gesichert")
note(f"Kommt der Film im Training vor, steigt die AP im Mittel um {d_known.mean():+.3f} (gepaarter t-Test über {len(d_known)} Falten, p = {t_known.pvalue:.3f}). "
     f"Ein Modell, das *nur* diesen Film kennt, ist gegenüber dem allgemeinen Modell um {d_only.mean():+.3f} AP {'besser' if d_only.mean() > 0 else 'schlechter'} (p = {t_only.pvalue:.3f}): "
     f"{'Wenige tausend Rezensionen eines Films ersetzen die allgemeine Spoiler-Sprache also nicht, ergänzen sie aber.' if d_only.mean() <= 0 else 'Für stark rezensierte Filme kann ein eigenes Modell also mithalten.'} "
     f"Das ist die Kehrseite von H2: Filmwissen hilft {'messbar' if t_known.pvalue < 0.05 else 'nicht gesichert'}, ist aber nicht die Hauptquelle der Erkennung.")

### 8.4 Abrufen: Abgleich mit der Handlungszusammenfassung

Die Präsentation nennt „Trainieren / Abrufen“. Für jeden Film gibt es eine Handlungszusammenfassung, auch für unbekannte Filme. Idee: Je ähnlicher eine Rezension der Handlung, desto eher verrät sie etwas. Gemessen als TF-IDF-Kosinusähnlichkeit (Stoppwörter entfernt), kombiniert per **Stacking auf der Validierungsmenge** (nie auf dem Training, wo das Textmodell die Labels fast auswendig kennt). Die Handlungstexte enthalten keine Labels: keine Leakage.

In [ ]:
plot_vec = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2, token_pattern=r"(?u)\b[a-zA-Z]{2,}\b")
plot_matrix = plot_vec.fit_transform(movies["plot_text"])
movie_pos = pd.Series(np.arange(len(movies)), index=movies["movie_id"].to_numpy())


def plot_similarity(df):
    rm = plot_vec.transform(df["review_text"])
    return np.asarray(rm.multiply(plot_matrix[movie_pos.loc[df["movie_id"]].to_numpy()]).sum(axis=1)).ravel()


parts = SPLITS["Nach Film"]
sim = {p: plot_similarity(parts[p]) for p in ("train", "val", "test")}
y_val, y_te = parts["val"]["label"].to_numpy(), parts["test"]["label"].to_numpy()
best_ev = test_preds["Nach Film"][best_name]
s_val_text = score_of(BEST["Nach Film"], parts["val"]["review_text"])

pred_sim = (sim["test"] >= best_threshold(y_val, sim["val"])).astype(int)
stacker = LogisticRegression(class_weight="balanced").fit(np.column_stack([s_val_text, sim["val"]]), y_val)
Z_test = np.column_stack([best_ev["scores"], sim["test"]])
s_stack = stacker.decision_function(Z_test)
variants = {"Nur Ähnlichkeit zur Handlung": (sim["test"], pred_sim), f"Nur Text ({best_name})": (best_ev["scores"], best_ev["preds"]),
            "Text + Ähnlichkeit (Stacking)": (s_stack, stacker.predict(Z_test))}
b = bootstrap_metrics(y_te, variants, n_boot=min(N_BOOT, 500))
rows = []
for name, (s, p) in variants.items():
    m = metrics(y_te, p, s)
    rows.append({"Variante": name, "F1": fmt_ci(m["F1"], *ci(b[name]["F1"])), "AP": fmt_ci(m["AP"], *ci(b[name]["AP"])), "ROC-AUC": m["ROC-AUC"]})
d = b["Text + Ähnlichkeit (Stacking)"]["AP"] - b[f"Nur Text ({best_name})"]["AP"]
lo, hi = ci(d)
retrieval = pd.DataFrame(rows).set_index("Variante")
display(save_table(retrieval, "08_abrufen"))

fig, ax = plt.subplots(figsize=(9, 3.8))
upper = np.quantile(sim["test"], 0.99)
bins = np.linspace(0, upper, 50)
for label, name, color in [(0, "Kein Spoiler", LIGHT), (1, "Spoiler", DARK)]:
    ax.hist(np.clip(sim["test"][y_te == label], 0, upper), bins=bins, density=True, alpha=0.65, color=color, label=name)
ax.set(xlabel="Kosinusähnlichkeit zur Handlungszusammenfassung", ylabel="Dichte", title="Ähnlichkeit zur Handlung je Klasse (Test, nach Film)")
ax.legend()
save_fig(fig, "08_abrufen")
findings["Abrufen (Handlungsabgleich)"] = (f"Ähnlichkeit allein AP {metrics(y_te, pred_sim, sim['test'])['AP']:.3f}; Stacking ΔAP = {d.mean():+.3f} [{lo:.3f}; {hi:.3f}] gegenüber nur Text",
                                           "Abgleich hilft signifikant" if lo > 0 else "Abgleich bringt keinen gesicherten Gewinn")
note(f"Die Ähnlichkeit allein trägt Information (AP {metrics(y_te, pred_sim, sim['test'])['AP']:.3f} > Zufall {y_te.mean():.3f}), "
     f"in Kombination mit dem Textmodell ändert sich die AP aber um {d.mean():+.3f} [{lo:.3f}; {hi:.3f}]. "
     f"{'Das Intervall enthält die Null: Die Ähnlichkeit enthält kaum Information, die nicht schon im Text steckt, weil beide auf denselben Handlungswörtern beruhen.' if lo <= 0 else 'Der Gewinn ist signifikant, aber klein; die Handlungszusammenfassung liefert nur wenig Information über den Text hinaus.'}")

### 8.5 Label-Qualität und Ablationen

Die Labels sind Selbstmarkierungen der Verfasser. Drei Prüfungen: (1) Widersprüche zwischen Text und Label („contains spoilers“, aber nicht markiert). (2) Ablation: Hauptmodell **mit** Warnmarkern (wie im früheren Ansatz) vs. ohne; und BoW statt TF-IDF. (3) Der Zeitverlauf (Kap. 3.2) als Hinweis auf wechselndes Markierungsverhalten.

In [ ]:
WARNING = re.compile(r"spoiler\s*(?:alert|warning)|spoilers?\s+(?:ahead|below|follow)|(?:contains?|has|with)\s+(?:some\s+|minor\s+|major\s+)?spoilers?"
                     r"|\(\s*spoilers?\s*\)|\*+\s*spoilers?", re.I)
NO_SPOILER = re.compile(r"\b(?:no|without)\s+(?:major\s+|any\s+|real\s+)?spoilers?\b|spoiler[\s-]*free", re.I)
warns, claims_none = sample["review_text"].str.contains(WARNING), sample["review_text"].str.contains(NO_SPOILER)
sample["Hinweis im Text"] = np.select([warns & claims_none, warns, claims_none], ["Beides", "Warnt vor Spoilern", "Behauptet: keine Spoiler"], default="Kein Hinweis")
label_check = sample.groupby("Hinweis im Text")["label"].agg(["size", "mean"])
label_check.columns = ["Rezensionen", "Davon als Spoiler markiert"]
display(save_table(label_check, "08_label_hinweise"))
warn_unlabeled = int((warns & ~claims_none & (sample["label"] == 0)).sum())
none_labeled = int((claims_none & ~warns & (sample["label"] == 1)).sum())
noise_share = warn_unlabeled / max(1, int((warns & ~claims_none).sum()))
print(f"Warnen im Text, aber NICHT als Spoiler markiert: {de(warn_unlabeled)} ({noise_share * 100:.0f} % der Warnungen)")
print(f"'Keine Spoiler' im Text, aber markiert:        {de(none_labeled)}")

In [ ]:
# Ablationen auf der Aufteilung nach Film (LogReg mit gewählten Hyperparametern)
parts = SPLITS["Nach Film"]
ablation = {}
_, ablation["TF-IDF ohne Warnmarker (Hauptmodell)"] = name_models[("Nach Film", "Original")]
_, ablation["TF-IDF mit Warnmarkern"] = fit_lr(parts, vectorizer=TfidfVectorizer(sublinear_tf=True, lowercase=True, **VECTORIZER_ARGS))
_, ablation["BoW (Zählungen) ohne Warnmarker"] = fit_lr(parts, vectorizer=CountVectorizer(preprocessor=preprocess, **VECTORIZER_ARGS))
y_te = parts["test"]["label"].to_numpy()
b = bootstrap_metrics(y_te, {k: (v["scores"], v["preds"]) for k, v in ablation.items()}, n_boot=min(N_BOOT, 500))
rows = []
for k, v in ablation.items():
    d = b[k]["AP"] - b["TF-IDF ohne Warnmarker (Hauptmodell)"]["AP"]
    lo, hi = ci(d)
    rows.append({"Variante": k, "F1": fmt_ci(v["F1"], *ci(b[k]["F1"])), "AP": fmt_ci(v["AP"], *ci(b[k]["AP"])),
                 "ΔAP zum Hauptmodell": "–" if k.startswith("TF-IDF ohne") else fmt_ci(float(d.mean()), lo, hi)})
ablation_table = pd.DataFrame(rows).set_index("Variante")
display(save_table(ablation_table, "08_ablationen"))
marker_gain = ablation["TF-IDF mit Warnmarkern"]["AP"] - ablation["TF-IDF ohne Warnmarker (Hauptmodell)"]["AP"]
findings["Label-Qualität"] = (f"{noise_share * 100:.0f} % der Rezensionen mit Warnhinweis sind nicht markiert; Warnmarker im Modell: ΔAP = {marker_gain:+.3f}",
                              "Labels enthalten dokumentierbares Rauschen; Warnmarker-Leakage ist klein und im Hauptmodell ausgeschlossen")
note(f"{noise_share * 100:.0f} % der Rezensionen, die ausdrücklich vor Spoilern warnen, sind **nicht** als Spoiler markiert: klare Label-Fehler, die als Fehlalarme des Modells gezählt werden. "
     f"Die gemessenen Werte sind deshalb eine **Untergrenze** der tatsächlichen Güte. Mit Warnmarkern im Text steigt die AP um {marker_gain:+.3f}: "
     f"Der frühere Ansatz hat davon profitiert, ohne es zu kontrollieren. TF-IDF ist gegenüber reinen Zählungen um "
     f"{ablation['TF-IDF ohne Warnmarker (Hauptmodell)']['AP'] - ablation['BoW (Zählungen) ohne Warnmarker']['AP']:+.3f} AP besser.")

### 8.6 Sensitivität: Aufteilung nach Nutzer

Kontrolle für Stil-Leakage (Fehleranalyse Nr. 9): Kein Nutzer in zwei Mengen. Fällt die Güte deutlich, hat das Modell Autorenstil gelernt.

In [ ]:
parts_user = split_by_group(sample, "user_id")
_, ev_user = fit_lr(parts_user)
ev_rand = name_models[("Zufällig", "Original")][1]
ev_film = name_models[("Nach Film", "Original")][1]
sens = pd.DataFrame({"F1": [ev_rand["F1"], ev_film["F1"], ev_user["F1"]], "AP": [ev_rand["AP"], ev_film["AP"], ev_user["AP"]],
                     "ROC-AUC": [ev_rand["ROC-AUC"], ev_film["ROC-AUC"], ev_user["ROC-AUC"]]},
                    index=["Zufällig", "Nach Film", "Nach Nutzer"]).rename_axis("Aufteilung")
display(save_table(sens, "08_nutzer_split"))
user_drop = ev_rand["AP"] - ev_user["AP"]
findings["Nutzer-Split (Stil-Leakage)"] = (f"AP nach Nutzer {ev_user['AP']:.3f} vs. zufällig {ev_rand['AP']:.3f} (Δ = {-user_drop:+.3f})",
                                            "kein Hinweis auf Stil-Leakage" if user_drop < 0.02 else "Autorenstil trägt messbar bei")
note(f"Ohne Nutzerüberlappung liegt die AP bei {ev_user['AP']:.3f} gegenüber {ev_rand['AP']:.3f} bei zufälliger Aufteilung (Δ = {-user_drop:+.3f}). "
     f"{'Der Unterschied liegt in der Größenordnung der Bootstrap-Intervalle; Autorenstil ist kein relevanter Leakage-Kanal.' if user_drop < 0.02 else 'Ein Teil der Güte stammt aus Autorenstil; die Hauptaussage (nach Film) bleibt davon unberührt, weil dort ebenfalls überwiegend neue Nutzer getestet werden.'}")

## 9 · Antwort auf die Forschungsfrage und Kernergebnisse

In [ ]:
ev_film = test_preds["Nach Film"][best_name]
ev_rand = test_preds["Zufällig"][best_name]
b_film, b_rand = boot["Nach Film"][best_name], boot["Zufällig"][best_name]
d_f1 = boot["Nach Film"][best_name]["F1"] - boot["Nach Film"]["Immer 'Spoiler'"]["F1"]
d_ap = boot["Nach Film"][best_name]["AP"] - boot["Nach Film"]["Nur Spoiler-Anteil des Films"]["AP"]
gain_f1 = float(np.mean(d_f1))
sig = ci(d_f1)[0] > 0 and ci(d_ap)[0] > 0
if not sig:
    strength = "Nein"
elif ev_film["AP"] >= 0.7:
    strength = "Ja, gut"
elif gain_f1 >= 0.08:
    strength = "Ja, aber nur teilweise"
else:
    strength = "Kaum"

answer = f'''
### Antwort: **{strength}.**

* **(a) Reicht die Wortwahl?** Ein Modell, das nur Wortvorkommen sieht und Warnmarker ignoriert, erreicht bei bekannten Filmen
  F1 = {fmt_ci(ev_rand["F1"], *ci(b_rand["F1"]))} und AP = {fmt_ci(ev_rand["AP"], *ci(b_rand["AP"]))}.
* **(b) Auch bei unbekannten Filmen?** Ohne jeden Testfilm im Training: F1 = {fmt_ci(ev_film["F1"], *ci(b_film["F1"]))},
  AP = {fmt_ci(ev_film["AP"], *ci(b_film["AP"]))}, ROC-AUC = {fmt_ci(ev_film["ROC-AUC"], *ci(b_film["ROC-AUC"]))} ({best_name}).
  Das ist {gain_f1:+.3f} F1 über „Immer Spoiler“ und {float(np.mean(d_ap)):+.3f} AP über dem Film-Prior; beide Differenzen sind
  {"signifikant (95-%-Bootstrap-Intervalle schließen die Null aus)" if sig else "nicht durchgängig signifikant"}.
* **Einordnung:** {"Die Erkennung ist praktisch brauchbar" if ev_film["AP"] >= 0.8 else "Die Erkennung funktioniert deutlich über Zufall, aber weit unter perfekt"}:
  Bei F1-optimaler Schwelle sind {ev_film["Precision"] * 100:.0f} % der Alarme berechtigt, und {100 - ev_film["Recall"] * 100:.0f} % der Spoiler werden übersehen.
  {"Die Wortwahl trägt ein klares, aber begrenztes Signal; ein Teil der Grenze ist Label-Rauschen (Kap. 8.5), ein Teil die Bag-of-Words-Darstellung, die Verneinung und Kontext nicht erfasst (H3)." if ev_film["AP"] < 0.8 else "Die verbleibenden Fehler gehen zum Teil auf Label-Rauschen zurück (Kap. 8.5)."}
* **Unbekannte Filme:** {findings["H2a · Unbekannte Filme sind schwerer"][1]} (Kap. 7.2); Figurennamen als Mechanismus: {findings["H2b · … weil Figurennamen fehlen"][1]} (Kap. 7.2b, 8.1).
  {"Das Modell lernt überwiegend *allgemeine* Spoiler-Sprache, die sich auf neue Filme überträgt." if abs(ev_rand["AP"] - ev_film["AP"]) < 0.05 else "Ein Teil der Erkennung hängt an filmspezifischem Vokabular."}
'''
display(Markdown(answer))
summary = pd.DataFrame([{"Frage": k, "Ergebnis": v[0], "Verdikt": v[1]} for k, v in findings.items()]).set_index("Frage")
with pd.option_context("display.max_colwidth", None):
    display(save_table(summary, "09_kernergebnisse"))

### Antworten auf die Diskussionsfragen der Präsentation

**Mehrere Modelle vergleichen oder auf eines konzentrieren?** Der Vergleich (Kap. 5, 6.4) zeigt, dass die linearen TF-IDF-Modelle statistisch kaum unterscheidbar sind und das Baum-Ensemble keinen Vorteil bringt. Für das Paper genügt ein Modell (logistische Regression wegen der interpretierbaren Gewichte); der Vergleich dient als Beleg, dass die Wahl das Ergebnis nicht treibt.

**Wie sehr können wir den Labels vertrauen?** Nur eingeschränkt (Kap. 8.5): Ein messbarer Anteil der Rezensionen mit ausdrücklicher Spoiler-Warnung ist nicht markiert, und der Spoiler-Anteil schwankt über die Jahre. Für den *Vergleich* von Verfahren ist das unproblematisch, weil alle mit denselben Labels arbeiten; absolute Werte sind Untergrenzen der Übereinstimmung mit dem Verfasserurteil, nicht mit einer objektiven Wahrheit.

## 10 · Limitationen

1. **Labels sind Selbstmarkierungen.** Kein unabhängiges Annotations-Goldstandard; Rauschen quantifiziert, aber nicht korrigiert. Eine Stichprobe manuell nachzuannotieren wäre der nächste Schritt.
2. **Stichprobe statt Vollerhebung.** 100.000 von ~574.000 Rezensionen; Konfidenzintervalle berücksichtigen das, aber seltene Genres sind dünn besetzt.
3. **Bag-of-Words ignoriert Reihenfolge und Verneinung.** „he doesn't die“ und „he dies“ sehen fast gleich aus (H3). Transformer-Modelle wären die natürliche Erweiterung; die Forschungsfrage zielt bewusst auf klassische Verfahren.
4. **Namensheuristik.** Figurennamen werden aus Handlungstexten mit einer Regel gewonnen; Schauspielernamen, Spitznamen und mehrdeutige Namen werden nicht vollständig erfasst. Da das Entfernen der erkannten Namen keinen Effekt hat, ist die Schlussfolgerung zu H2b robust gegenüber einer unvollständigen Liste, aber nicht gegenüber einer systematisch falschen.
5. **Wortlisten für H1** sind klein und von den Autoren gewählt (vorab fixiert, im Code dokumentiert). Ein externes Lexikon wäre objektiver.
6. **Wenige Seeds für H2** (`N_SEEDS`): Der t-Test ist entsprechend konservativ; die Konsistenz der Richtung wird deshalb mitberichtet.
7. **Ein Datensatz, eine Plattform (IMDb), Englisch.** Übertragbarkeit auf andere Sprachen oder Plattformen ist nicht geprüft.
8. **Schwellenwahl auf der Validierungsmenge** optimiert F1; andere Anwendungen (z. B. Warnhinweise mit hohem Recall) brauchen andere Schwellen.

## 11 · Gutachter-Checkliste

| Was ein strenger Gutachter fragen würde | Wo es adressiert ist |
|---|---|
| Gibt es Leakage zwischen Training und Test? | Textduplikate entfernt (2.2); Vektorisierung/SVD nur in Pipeline auf Trainingsfalten (4.2); Film-Gruppen auch innerhalb der CV (4.3); Namenslisten und Handlungstexte labelfrei (2.3, 8.4); Stacking nur auf Validierung (8.4); Nutzer-Split als Sensitivität (8.6) |
| Steckt die Zielvariable in den Merkmalen? | Warnmarker in der Pipeline entfernt; Effekt als Ablation quantifiziert (8.5) |
| Ist die Metrik angemessen? | Accuracy verworfen (3.2); F1 + AP + ROC-AUC mit Begründung (4.3) |
| Wurde auf der Testmenge optimiert? | Nein: Hyperparameter per CV, Schwelle und Modellauswahl auf Validierung (4.3, 5.1) |
| Sind die Unterschiede signifikant? | Gepaarter Bootstrap für alle Vergleiche (6.3, 6.4, 8.x); t-Test/Wilcoxon über Seeds (7.2, 8.3) |
| Ist das Ergebnis nur Textlänge? | Nur-Länge-Baseline und Längenquartile (6.6); H3 längenadjustiert (7.3) |
| Sind die Hypothesen falsifizierbar formuliert? | Prüfverfahren und Verdiktregel vor jedem Test festgelegt (7.x) |
| Wurden Hypothesen nach den Daten angepasst? | H1-Wortlisten und Alarmwörter vorab definiert; H2 in Effekt und Mechanismus getrennt, beide berichtet, auch wenn der Mechanismus scheitert |
| Reproduzierbarkeit? | Feste Seeds überall, inkl. `LinearSVC`; Cache; Konfiguration an einer Stelle (Kap. 0) |
| Mehrfachvergleiche? | Viele Tests werden berichtet; die Hauptaussage stützt sich auf zwei vorab festgelegte Vergleiche (6.3). Nebenbefunde (Genres, Filme) sind als explorativ gekennzeichnet |
| Baum-Ensemble auf Text fair eingesetzt? | SVD-Reduktion begründet (4.2); Ergebnis als Ergänzung, nicht als Hauptmodell |
| Was bleibt offen? | Kap. 10 |

## 12 · Exportierte Abbildungen und Tabellen

In [ ]:
export_list = pd.DataFrame({"Datei": sorted(set(exports))})
export_list["Typ"] = export_list["Datei"].str.split(".").str[-1].str.upper()
print(f"{(export_list['Typ'] == 'PNG').sum()} Abbildungen (300 dpi) in '{FIG_DIR}/', {(export_list['Typ'] == 'CSV').sum()} Tabellen in '{TAB_DIR}/'")
export_list